# ASCAT + Recon 单文件运行 Notebook（CDS JupyterLab）

这个 Notebook 的目标是：
- 不依赖在 CDS 上部署整个项目；
- 只通过一个 `.ipynb` 完成 ASCAT/Recon 任务提交、年度断点续跑、结果合并；
- 仅输出轻量 `features.csv + summary.json`。

## 你需要准备的输入

两种模式：

1. **推荐（无需 NOAA 原始目录）**：在本地先生成 manifest，上传到 CDS：
- `ascat_request_manifest.csv`
- `recon_request_manifest.csv`

2. **可选**：如果你把 `noaa/` 和 `storm_id_crosswalk.csv` 也上传到 CDS，则可在 Notebook 内生成 manifest。

按顺序执行所有单元格即可。

## 运行顺序

请按顺序运行，或直接 **Run All**：
1. Step 0 依赖安装
2. CONFIG 初始化
3. run() helper
4. embedded_scripts 写入
5. 工具函数
6. Step 1 / Step 2 / Step 3


In [ ]:
# Step 0: install runtime dependencies (run this first in a fresh kernel)
import sys
import subprocess


def _pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs]
    print("[RUN]", " ".join(cmd))
    subprocess.check_call(cmd)


_pip_install(["pip"])
_pip_install(["numpy", "xarray", "copernicusmarine"])
print("dependency installation done")

In [ ]:
from pathlib import Path
import os
import json
import csv
import subprocess
from datetime import datetime

CONFIG = {
    # 运行根目录（Notebook 会在这里写临时脚本和输出）
    "workdir": "/home/jovyan/mystorage/cds_obs_pipeline_runtime",

    # 依赖安装已在上一个单元完成

    # 是否在 Notebook 内构建 manifest
    # False: 直接使用你上传的 manifest 文件
    # True : 需要 noaa_root + crosswalk_csv
    "build_manifest_in_notebook": False,

    # 仅当 build_manifest_in_notebook=True 时使用
    "noaa_root": "/home/jovyan/mystorage/noaa",
    "crosswalk_csv": "/home/jovyan/mystorage/storm_id_crosswalk.csv",

    # 处理年份区间
    "year_start": 2016,
    "year_end": 2016,

    # 若 build_manifest_in_notebook=False，填写你上传的 manifest 路径
    "ascat_manifest_input": "/home/jovyan/mystorage/ascat_request_manifest.csv",
    "recon_manifest_input": "/home/jovyan/mystorage/recon_request_manifest.csv",

    # ASCAT 认证（优先级：CONFIG > 环境变量）
    # 环境变量名：COPERNICUSMARINE_SERVICE_USERNAME / COPERNICUSMARINE_SERVICE_PASSWORD
    "ascat_username": "",
    "ascat_password": "",
    "ascat_credentials_file": "",
    "ascat_service": "",
    # 若为 False 且未提供非交互凭据，会直接报错而不是卡在交互输入
    "allow_interactive_credential_prompt": False,

    # 运行开关
    "run_ascat": True,
    "run_recon": True,

    # 过滤与重试参数
    "only_with_storm_id": True,
    "ascat_max_rows": 0,
    "recon_max_rows": 0,
    "ascat_max_retries": 3,
    "recon_max_retries": 3,
    "ascat_sleep_sec": 0.2,
    "recon_sleep_sec": 0.1,

    # 低内存保护：按年份内分块执行，块完成后可清理中间文件
    "ascat_chunk_rows": 5,
    "recon_chunk_rows": 50,
    "cleanup_chunk_intermediate": True,
    "cleanup_ascat_tmp_dir": True,
    "ascat_tmp_dir": "/tmp/ascat_subset_tmp",
        "safe_mode_note": "run one year per session on 4CPU/16GB pod",
        "ascat_window_before_min": 120,
        "ascat_window_after_min": 120,
        "ascat_outer_radius_km": 350.0,
        "ascat_bbox_margin_deg": 0.2,
}

WORKDIR = Path(CONFIG["workdir"]).resolve()
SCRIPTS_DIR = WORKDIR / "scripts"
MANIFEST_DIR = WORKDIR / "manifests"
ASCAT_DIR = WORKDIR / "ascat"
RECON_DIR = WORKDIR / "recon"

for p in [WORKDIR, SCRIPTS_DIR, MANIFEST_DIR, ASCAT_DIR, RECON_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("WORKDIR:", WORKDIR)

In [ ]:
def run(cmd, check=True):
    print("\n[RUN]", " ".join(cmd))
    proc = subprocess.run(cmd, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd)}")

print("run() helper ready")


In [ ]:
from pathlib import Path

embedded_scripts = {
    "build_obs_request_manifest.py": "#!/usr/bin/env python3\n\"\"\"Build ASCAT/Recon request manifests from NOAA forecast advisory files.\n\nThe script follows the same manifest style as GOES:\n- one row per advisory issue cycle\n- point-time anchor from advisory center location\n- optional storm_id backfill via ATCF crosswalk\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport re\nfrom dataclasses import dataclass\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Set, Tuple\n\n\nMONTH_TO_NUM = {\n    \"JAN\": 1,\n    \"FEB\": 2,\n    \"MAR\": 3,\n    \"APR\": 4,\n    \"MAY\": 5,\n    \"JUN\": 6,\n    \"JUL\": 7,\n    \"AUG\": 8,\n    \"SEP\": 9,\n    \"OCT\": 10,\n    \"NOV\": 11,\n    \"DEC\": 12,\n}\n\nISSUE_TIME_RE = re.compile(\n    r\"^\\s*(\\d{3,4})\\s+UTC\\s+[A-Z]{3}\\s+([A-Z]{3})\\s+(\\d{1,2})\\s+(\\d{4})\\s*$\",\n    re.IGNORECASE,\n)\nCENTER_RE = re.compile(\n    r\"CENTER LOCATED NEAR\\s+([0-9.]+)([NS])\\s+([0-9.]+)([EW])\\s+AT\\s+(\\d{1,2})/(\\d{4})Z\",\n    re.IGNORECASE,\n)\nATCF_ID_RE = re.compile(r\"\\b([A-Z]{2}\\d{2}\\d{4})\\b\")\nADV_NO_RE = re.compile(r\"FORECAST/ADVISORY NUMBER\\s+(\\d+)\", re.IGNORECASE)\n\nOUT_FIELDS = [\n    \"request_id\",\n    \"storm_id\",\n    \"storm_id_match_status\",\n    \"atcf_storm_id\",\n    \"basin\",\n    \"storm_name\",\n    \"advisory_no\",\n    \"issue_time_utc\",\n    \"center_obs_day\",\n    \"center_obs_hhmmz\",\n    \"lat\",\n    \"lon\",\n    \"source_file\",\n    \"is_recon_candidate\",\n]\n\n\n@dataclass\nclass ManifestRow:\n    request_id: str\n    storm_id: str\n    storm_id_match_status: str\n    atcf_storm_id: str\n    basin: str\n    storm_name: str\n    advisory_no: str\n    issue_time_utc: str\n    center_obs_day: str\n    center_obs_hhmmz: str\n    lat: float\n    lon: float\n    source_file: str\n    is_recon_candidate: int\n\n    def to_dict(self) -> Dict[str, object]:\n        return {\n            \"request_id\": self.request_id,\n            \"storm_id\": self.storm_id,\n            \"storm_id_match_status\": self.storm_id_match_status,\n            \"atcf_storm_id\": self.atcf_storm_id,\n            \"basin\": self.basin,\n            \"storm_name\": self.storm_name,\n            \"advisory_no\": self.advisory_no,\n            \"issue_time_utc\": self.issue_time_utc,\n            \"center_obs_day\": self.center_obs_day,\n            \"center_obs_hhmmz\": self.center_obs_hhmmz,\n            \"lat\": self.lat,\n            \"lon\": self.lon,\n            \"source_file\": self.source_file,\n            \"is_recon_candidate\": self.is_recon_candidate,\n        }\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=\"Build ASCAT + Recon request manifests from NOAA advisory files.\"\n    )\n    parser.add_argument(\n        \"--noaa-root\",\n        type=Path,\n        default=Path(\"noaa\"),\n        help=\"NOAA root folder.\",\n    )\n    parser.add_argument(\n        \"--crosswalk-csv\",\n        type=Path,\n        default=Path(\"data/interim/atcf/storm_id_crosswalk.csv\"),\n        help=\"ATCF storm crosswalk csv.\",\n    )\n    parser.add_argument(\n        \"--ascat-out-csv\",\n        type=Path,\n        default=Path(\"data/interim/ascat/ascat_request_manifest.csv\"),\n        help=\"ASCAT manifest output path.\",\n    )\n    parser.add_argument(\n        \"--recon-out-csv\",\n        type=Path,\n        default=Path(\"data/interim/recon/recon_request_manifest.csv\"),\n        help=\"Recon manifest output path.\",\n    )\n    parser.add_argument(\n        \"--year-start\",\n        type=int,\n        default=2016,\n        help=\"Inclusive year lower bound from NOAA folder.\",\n    )\n    parser.add_argument(\n        \"--year-end\",\n        type=int,\n        default=2025,\n        help=\"Inclusive year upper bound from NOAA folder.\",\n    )\n    parser.add_argument(\n        \"--recon-basins\",\n        type=str,\n        default=\"Atlantic,E_Pacific\",\n        help=\"Comma separated basin folder names considered recon-priority.\",\n    )\n    parser.add_argument(\n        \"--keep-all-recon-rows\",\n        action=\"store_true\",\n        help=\"If set, keep all rows in recon manifest and use is_recon_candidate as a flag only.\",\n    )\n    parser.add_argument(\n        \"--only-with-storm-id\",\n        action=\"store_true\",\n        help=\"Keep only rows with non-empty storm_id in both manifests.\",\n    )\n    parser.add_argument(\n        \"--limit\",\n        type=int,\n        default=0,\n        help=\"If >0, keep first N rows (after sorting) for each manifest.\",\n    )\n    return parser.parse_args()\n\n\ndef parse_issue_time(lines: List[str]) -> Optional[datetime]:\n    for ln in lines[:50]:\n        m = ISSUE_TIME_RE.match(ln.strip().upper())\n        if not m:\n            continue\n        hhmm_raw = m.group(1).zfill(4)\n        month_txt = m.group(2).upper()\n        day = int(m.group(3))\n        year = int(m.group(4))\n        month = MONTH_TO_NUM.get(month_txt)\n        if month is None:\n            continue\n        try:\n            return datetime(\n                year=year,\n                month=month,\n                day=day,\n                hour=int(hhmm_raw[:2]),\n                minute=int(hhmm_raw[2:]),\n            )\n        except ValueError:\n            continue\n    return None\n\n\ndef parse_center(lines: List[str]) -> Optional[Tuple[float, float, str, str]]:\n    for ln in lines:\n        m = CENTER_RE.search(ln)\n        if not m:\n            continue\n        lat = float(m.group(1)) * (1.0 if m.group(2).upper() == \"N\" else -1.0)\n        lon = float(m.group(3)) * (-1.0 if m.group(4).upper() == \"W\" else 1.0)\n        obs_day = m.group(5).zfill(2)\n        obs_hhmmz = m.group(6).zfill(4)\n        return lat, lon, obs_day, obs_hhmmz\n    return None\n\n\ndef parse_atcf_id(lines: List[str]) -> str:\n    for ln in lines[:40]:\n        m = ATCF_ID_RE.search(ln)\n        if m:\n            return m.group(1).upper()\n    return \"\"\n\n\ndef parse_advisory_no(lines: List[str]) -> str:\n    for ln in lines[:60]:\n        m = ADV_NO_RE.search(ln)\n        if not m:\n            continue\n        try:\n            return f\"{int(m.group(1)):03d}\"\n        except Exception:\n            return m.group(1).strip()\n    return \"\"\n\n\ndef load_crosswalk(path: Path) -> Dict[str, Tuple[str, str]]:\n    if not path.exists():\n        return {}\n    out: Dict[str, Tuple[str, str]] = {}\n    with path.open(\"r\", encoding=\"utf-8\", newline=\"\") as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n            atcf = (row.get(\"atcf_storm_id\") or \"\").strip().upper()\n            if not atcf:\n                continue\n            matched = (row.get(\"matched_storm_id\") or \"\").strip()\n            status = (row.get(\"match_status\") or \"\").strip() or \"unknown\"\n            out[atcf] = (matched, status)\n    return out\n\n\ndef iter_forecast_advisory_files(noaa_root: Path) -> Iterable[Path]:\n    return sorted(noaa_root.glob(\"*/*/*/forecast_advisory/*.txt\"))\n\n\ndef year_from_path(path: Path, noaa_root: Path) -> Optional[int]:\n    try:\n        rel = path.relative_to(noaa_root)\n    except ValueError:\n        return None\n    if not rel.parts:\n        return None\n    try:\n        return int(rel.parts[0])\n    except Exception:\n        return None\n\n\ndef parse_recon_basins(value: str) -> Set[str]:\n    out: Set[str] = set()\n    for item in (value or \"\").split(\",\"):\n        token = item.strip()\n        if token:\n            out.add(token)\n    return out\n\n\ndef build_rows(args: argparse.Namespace) -> Tuple[List[ManifestRow], Dict[str, int]]:\n    crosswalk = load_crosswalk(args.crosswalk_csv)\n    recon_basins = parse_recon_basins(args.recon_basins)\n    rows: List[ManifestRow] = []\n    stats: Dict[str, int] = {\n        \"files_seen\": 0,\n        \"rows_built\": 0,\n        \"skip_issue_time\": 0,\n        \"skip_center\": 0,\n        \"skip_year_filter\": 0,\n        \"with_storm_id_match\": 0,\n        \"without_storm_id_match\": 0,\n        \"recon_candidate_rows\": 0,\n    }\n\n    for file_path in iter_forecast_advisory_files(args.noaa_root):\n        stats[\"files_seen\"] += 1\n        yr = year_from_path(file_path, args.noaa_root)\n        if yr is None or yr < args.year_start or yr > args.year_end:\n            stats[\"skip_year_filter\"] += 1\n            continue\n\n        lines = file_path.read_text(encoding=\"utf-8\", errors=\"ignore\").splitlines()\n        issue_dt = parse_issue_time(lines)\n        if issue_dt is None:\n            stats[\"skip_issue_time\"] += 1\n            continue\n\n        center = parse_center(lines)\n        if center is None:\n            stats[\"skip_center\"] += 1\n            continue\n        lat, lon, obs_day, obs_hhmmz = center\n\n        atcf_storm_id = parse_atcf_id(lines)\n        advisory_no = parse_advisory_no(lines)\n        rel = file_path.relative_to(args.noaa_root)\n        basin = rel.parts[1] if len(rel.parts) >= 2 else \"\"\n        storm_name = rel.parts[2] if len(rel.parts) >= 3 else \"\"\n\n        storm_id = \"\"\n        match_status = \"unresolved\"\n        if atcf_storm_id and atcf_storm_id in crosswalk:\n            storm_id, match_status = crosswalk[atcf_storm_id]\n            if storm_id:\n                stats[\"with_storm_id_match\"] += 1\n            else:\n                stats[\"without_storm_id_match\"] += 1\n        else:\n            stats[\"without_storm_id_match\"] += 1\n\n        is_recon_candidate = 1 if basin in recon_basins else 0\n        if is_recon_candidate:\n            stats[\"recon_candidate_rows\"] += 1\n\n        if args.only_with_storm_id and not storm_id:\n            continue\n\n        issue_time_utc = issue_dt.strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n        request_id = (\n            f\"{basin}_{atcf_storm_id or storm_name}_{issue_dt.strftime('%Y%m%dT%H%M')}_{advisory_no or 'UNK'}\"\n        )\n\n        rows.append(\n            ManifestRow(\n                request_id=request_id,\n                storm_id=storm_id,\n                storm_id_match_status=match_status,\n                atcf_storm_id=atcf_storm_id,\n                basin=basin,\n                storm_name=storm_name,\n                advisory_no=advisory_no,\n                issue_time_utc=issue_time_utc,\n                center_obs_day=obs_day,\n                center_obs_hhmmz=obs_hhmmz,\n                lat=lat,\n                lon=lon,\n                source_file=str(file_path),\n                is_recon_candidate=is_recon_candidate,\n            )\n        )\n        stats[\"rows_built\"] += 1\n\n    rows.sort(key=lambda x: (x.issue_time_utc, x.basin, x.storm_name, x.request_id))\n    return rows, stats\n\n\ndef write_csv(path: Path, rows: List[ManifestRow]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\") as f:\n        writer = csv.DictWriter(f, fieldnames=OUT_FIELDS)\n        writer.writeheader()\n        for row in rows:\n            writer.writerow(row.to_dict())\n\n\ndef apply_limit(rows: List[ManifestRow], limit: int) -> List[ManifestRow]:\n    if limit <= 0:\n        return rows\n    return rows[:limit]\n\n\ndef main() -> int:\n    args = parse_args()\n    rows, stats = build_rows(args)\n\n    ascat_rows = apply_limit(rows, args.limit)\n    recon_rows_all = rows if args.keep_all_recon_rows else [r for r in rows if r.is_recon_candidate == 1]\n    recon_rows = apply_limit(recon_rows_all, args.limit)\n\n    write_csv(args.ascat_out_csv, ascat_rows)\n    write_csv(args.recon_out_csv, recon_rows)\n\n    print(\"ascat_manifest:\", args.ascat_out_csv)\n    print(\"recon_manifest:\", args.recon_out_csv)\n    print(\"rows_total_built:\", len(rows))\n    print(\"ascat_rows_written:\", len(ascat_rows))\n    print(\"recon_rows_written:\", len(recon_rows))\n    print(\"recon_candidate_rows_total:\", sum(1 for r in rows if r.is_recon_candidate == 1))\n    print(\"rows_with_storm_id:\", sum(1 for r in rows if r.storm_id))\n    print(\"rows_without_storm_id:\", sum(1 for r in rows if not r.storm_id))\n    print(\"files_seen:\", stats[\"files_seen\"])\n    print(\"skip_year_filter:\", stats[\"skip_year_filter\"])\n    print(\"skip_issue_time:\", stats[\"skip_issue_time\"])\n    print(\"skip_center:\", stats[\"skip_center\"])\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
    "extract_ascat_features_remote.py": "#!/usr/bin/env python3\n\"\"\"Extract ASCAT structured wind features with remote subsetting on Copernicus Marine.\n\nDesign goals:\n1. Keep raw data request and heavy handling on remote compute nodes.\n2. Save only compact per-request feature rows to local CSV.\n3. Keep failure-tolerant behavior (one failed request should not stop the run).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport inspect\nimport json\nimport math\nimport time\nfrom dataclasses import dataclass\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\n\nDEFAULT_DATASET_IDS = [\n    \"cmems_obs-wind_glo_phy-ascat-metop_a-l3-pt1h\",\n    \"cmems_obs-wind_glo_phy-ascat-metop_b-l3-pt1h\",\n    \"cmems_obs-wind_glo_phy-ascat-metop_c-l3-pt1h\",\n]\n\nOUTPUT_FIELDS = [\n    \"request_id\",\n    \"storm_id\",\n    \"storm_id_match_status\",\n    \"atcf_storm_id\",\n    \"basin\",\n    \"storm_name\",\n    \"advisory_no\",\n    \"issue_time_utc\",\n    \"lat\",\n    \"lon\",\n    \"source_file\",\n    \"ascat_status\",\n    \"missing_reason\",\n    \"obs_time_utc\",\n    \"obs_offset_minutes\",\n    \"obs_offset_abs_minutes\",\n    \"ascat_dataset_id\",\n    \"ascat_platform\",\n    \"ascat_variable\",\n    \"ascat_units\",\n    \"inner_radius_km\",\n    \"outer_radius_km\",\n    \"wind_mean_inner_kt\",\n    \"wind_p90_inner_kt\",\n    \"wind_max_inner_kt\",\n    \"wind_mean_ring_kt\",\n    \"wind_p90_ring_kt\",\n    \"wind_max_ring_kt\",\n    \"wind_area_ge34kt_inner_km2\",\n    \"wind_area_ge50kt_inner_km2\",\n    \"valid_cell_count\",\n    \"qc_has_data\",\n    \"qc_time_within_window\",\n]\n\n\n@dataclass\nclass RequestRow:\n    request_id: str\n    issue_time_utc: str\n    issue_dt: datetime\n    lat: float\n    lon: float\n    raw: Dict[str, str]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=\"Extract ASCAT compact feature table from request manifest via Copernicus Marine.\"\n    )\n    parser.add_argument(\n        \"--manifest-csv\",\n        type=Path,\n        default=Path(\"data/interim/ascat/ascat_request_manifest.csv\"),\n        help=\"Input request manifest csv.\",\n    )\n    parser.add_argument(\n        \"--out-csv\",\n        type=Path,\n        default=Path(\"data/interim/ascat/ascat_observation_features.csv\"),\n        help=\"Output feature csv.\",\n    )\n    parser.add_argument(\n        \"--summary-json\",\n        type=Path,\n        default=Path(\"data/interim/ascat/ascat_observation_features_summary.json\"),\n        help=\"Output summary json.\",\n    )\n    parser.add_argument(\n        \"--dataset-id\",\n        action=\"append\",\n        dest=\"dataset_ids\",\n        default=[],\n        help=\"Repeatable Copernicus Marine dataset id.\",\n    )\n    parser.add_argument(\n        \"--variable\",\n        action=\"append\",\n        dest=\"variables\",\n        default=[],\n        help=\"Optional variable filters passed to subset call.\",\n    )\n    parser.add_argument(\n        \"--window-before-min\",\n        type=int,\n        default=180,\n        help=\"Look-back window before issue time.\",\n    )\n    parser.add_argument(\n        \"--window-after-min\",\n        type=int,\n        default=180,\n        help=\"Look-forward window after issue time.\",\n    )\n    parser.add_argument(\n        \"--inner-radius-km\",\n        type=float,\n        default=200.0,\n        help=\"Inner disk radius for core stats.\",\n    )\n    parser.add_argument(\n        \"--outer-radius-km\",\n        type=float,\n        default=500.0,\n        help=\"Outer disk radius for ring stats.\",\n    )\n    parser.add_argument(\n        \"--bbox-margin-deg\",\n        type=float,\n        default=0.5,\n        help=\"Extra bbox degree margin around outer-radius projection.\",\n    )\n    parser.add_argument(\n        \"--assumed-cell-km\",\n        type=float,\n        default=25.0,\n        help=\"Fallback grid-cell edge size for area estimation when resolution is unknown.\",\n    )\n    parser.add_argument(\n        \"--tmp-dir\",\n        type=Path,\n        default=Path(\"/tmp/ascat_subset_tmp\"),\n        help=\"Temporary folder for subset netcdf files.\",\n    )\n    parser.add_argument(\n        \"--keep-temp-files\",\n        action=\"store_true\",\n        help=\"Keep temporary subset files.\",\n    )\n    parser.add_argument(\n        \"--max-rows\",\n        type=int,\n        default=0,\n        help=\"If >0, process only first N manifest rows.\",\n    )\n    parser.add_argument(\n        \"--only-with-storm-id\",\n        action=\"store_true\",\n        help=\"Keep only rows with non-empty storm_id.\",\n    )\n    parser.add_argument(\n        \"--max-retries\",\n        type=int,\n        default=3,\n        help=\"Retries for subset calls per dataset request.\",\n    )\n    parser.add_argument(\n        \"--sleep-sec\",\n        type=float,\n        default=0.2,\n        help=\"Sleep between requests to reduce throttling.\",\n    )\n    parser.add_argument(\n        \"--username\",\n        type=str,\n        default=\"\",\n        help=\"Copernicus Marine username (optional, fallback to env/config).\",\n    )\n    parser.add_argument(\n        \"--password\",\n        type=str,\n        default=\"\",\n        help=\"Copernicus Marine password (optional, fallback to env/config).\",\n    )\n    parser.add_argument(\n        \"--credentials-file\",\n        type=str,\n        default=\"\",\n        help=\"Optional credentials file path passed to subset call if supported.\",\n    )\n    parser.add_argument(\n        \"--service\",\n        type=str,\n        default=\"\",\n        help=\"Optional service endpoint passed to subset call if supported.\",\n    )\n    parser.add_argument(\n        \"--dry-run\",\n        action=\"store_true\",\n        help=\"Only print resolved config and row counts.\",\n    )\n    return parser.parse_args()\n\n\ndef parse_iso_utc(value: str) -> datetime:\n    txt = (value or \"\").strip()\n    fmts = [\n        \"%Y-%m-%dT%H:%M:%SZ\",\n        \"%Y-%m-%d %H:%M:%S\",\n        \"%Y-%m-%dT%H:%M:%S\",\n        \"%Y-%m-%d %H:%M:%S.%f\",\n        \"%Y-%m-%dT%H:%M:%S.%f\",\n    ]\n    for fmt in fmts:\n        try:\n            return datetime.strptime(txt, fmt).replace(tzinfo=timezone.utc)\n        except ValueError:\n            continue\n    raise ValueError(f\"unsupported datetime format: {value}\")\n\n\ndef parse_float(value: Any) -> Optional[float]:\n    try:\n        if value is None:\n            return None\n        txt = str(value).strip()\n        if not txt:\n            return None\n        return float(txt)\n    except Exception:\n        return None\n\n\ndef safe_request_id(request_id: str) -> str:\n    out = []\n    for ch in request_id:\n        if ch.isalnum() or ch in {\"-\", \"_\"}:\n            out.append(ch)\n        else:\n            out.append(\"_\")\n    return \"\".join(out)[:128]\n\n\ndef parse_lon_minus180_180(lon: float) -> float:\n    x = lon\n    while x >= 180:\n        x -= 360\n    while x < -180:\n        x += 360\n    return x\n\n\ndef build_bbox(lat: float, lon: float, outer_radius_km: float, margin_deg: float) -> Tuple[float, float, float, float]:\n    lat_delta = (outer_radius_km / 111.0) + margin_deg\n    cos_lat = max(0.15, abs(math.cos(math.radians(lat))))\n    lon_delta = (outer_radius_km / (111.0 * cos_lat)) + margin_deg\n    lon_c = parse_lon_minus180_180(lon)\n    min_lon = lon_c - lon_delta\n    max_lon = lon_c + lon_delta\n\n    if min_lon < -180:\n        min_lon += 360\n        max_lon += 360\n    if max_lon > 360:\n        max_lon = 360.0\n    if min_lon < -180:\n        min_lon = -180.0\n\n    min_lat = max(-89.9, lat - lat_delta)\n    max_lat = min(89.9, lat + lat_delta)\n    return min_lon, max_lon, min_lat, max_lat\n\n\ndef load_manifest(args: argparse.Namespace) -> List[RequestRow]:\n    if not args.manifest_csv.exists():\n        raise FileNotFoundError(f\"manifest not found: {args.manifest_csv}\")\n\n    rows: List[RequestRow] = []\n    with args.manifest_csv.open(\"r\", encoding=\"utf-8\", newline=\"\") as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n            issue_time = (row.get(\"issue_time_utc\") or \"\").strip()\n            lat = parse_float(row.get(\"lat\"))\n            lon = parse_float(row.get(\"lon\"))\n            if not issue_time or lat is None or lon is None:\n                continue\n            if args.only_with_storm_id and not (row.get(\"storm_id\") or \"\").strip():\n                continue\n            try:\n                issue_dt = parse_iso_utc(issue_time)\n            except ValueError:\n                continue\n            request_id = (row.get(\"request_id\") or \"\").strip()\n            if not request_id:\n                request_id = f\"REQ_{issue_dt.strftime('%Y%m%dT%H%M%S')}_{len(rows):06d}\"\n            rows.append(\n                RequestRow(\n                    request_id=request_id,\n                    issue_time_utc=issue_dt.strftime(\"%Y-%m-%dT%H:%M:%SZ\"),\n                    issue_dt=issue_dt,\n                    lat=float(lat),\n                    lon=float(lon),\n                    raw={k: (v or \"\").strip() for k, v in row.items()},\n                )\n            )\n    rows.sort(key=lambda x: (x.issue_time_utc, x.request_id))\n    if args.max_rows > 0:\n        rows = rows[: args.max_rows]\n    return rows\n\n\ndef init_optional_deps() -> Tuple[Any, Any, Any]:\n    try:\n        import numpy as np  # type: ignore\n    except Exception as exc:  # pragma: no cover - import guard\n        raise RuntimeError(\"numpy is required for ASCAT feature extraction\") from exc\n    try:\n        import xarray as xr  # type: ignore\n    except Exception as exc:  # pragma: no cover - import guard\n        raise RuntimeError(\"xarray is required for ASCAT feature extraction\") from exc\n    try:\n        import copernicusmarine  # type: ignore\n    except Exception as exc:  # pragma: no cover - import guard\n        raise RuntimeError(\n            \"copernicusmarine package is required on remote node. \"\n            \"Install via: pip install copernicusmarine\"\n        ) from exc\n    return np, xr, copernicusmarine\n\n\ndef call_subset(\n    copernicusmarine: Any,\n    dataset_id: str,\n    out_file: Path,\n    min_lon: float,\n    max_lon: float,\n    min_lat: float,\n    max_lat: float,\n    start_dt: datetime,\n    end_dt: datetime,\n    args: argparse.Namespace,\n) -> Path:\n    subset_sig = inspect.signature(copernicusmarine.subset)\n    accepted = set(subset_sig.parameters.keys())\n\n    kwargs: Dict[str, Any] = {\n        \"dataset_id\": dataset_id,\n        \"minimum_longitude\": float(min_lon),\n        \"maximum_longitude\": float(max_lon),\n        \"minimum_latitude\": float(min_lat),\n        \"maximum_latitude\": float(max_lat),\n        \"start_datetime\": start_dt.strftime(\"%Y-%m-%dT%H:%M:%SZ\"),\n        \"end_datetime\": end_dt.strftime(\"%Y-%m-%dT%H:%M:%SZ\"),\n        \"output_directory\": str(out_file.parent),\n        \"output_filename\": out_file.name,\n        \"overwrite\": True,\n        \"disable_progress_bar\": True,\n    }\n    if args.variables:\n        kwargs[\"variables\"] = args.variables\n    if args.username:\n        kwargs[\"username\"] = args.username\n    if args.password:\n        kwargs[\"password\"] = args.password\n    if args.credentials_file:\n        kwargs[\"credentials_file\"] = args.credentials_file\n    if args.service:\n        kwargs[\"service\"] = args.service\n\n    filtered_kwargs = {k: v for k, v in kwargs.items() if k in accepted and v is not None and v != \"\"}\n    result = copernicusmarine.subset(**filtered_kwargs)\n\n    if out_file.exists() and out_file.stat().st_size > 0:\n        return out_file\n\n    # Fallback: try to resolve file path from response object.\n    if isinstance(result, dict):\n        for key in (\"output_path\", \"file_path\", \"path\"):\n            if key in result and result[key]:\n                cand = Path(str(result[key]))\n                if cand.exists() and cand.stat().st_size > 0:\n                    return cand\n        if \"files\" in result and isinstance(result[\"files\"], list):\n            for fp in result[\"files\"]:\n                cand = Path(str(fp))\n                if cand.exists() and cand.stat().st_size > 0:\n                    return cand\n\n    raise RuntimeError(f\"subset call finished but output file is missing: {out_file}\")\n\n\ndef to_datetime_list(np: Any, values: Any) -> List[datetime]:\n    out: List[datetime] = []\n    arr = np.asarray(values)\n    for v in arr.reshape(-1):\n        try:\n            if isinstance(v, datetime):\n                out.append(v.replace(tzinfo=timezone.utc) if v.tzinfo is None else v.astimezone(timezone.utc))\n                continue\n            if isinstance(v, str):\n                out.append(parse_iso_utc(v))\n                continue\n            if np.issubdtype(type(v), np.datetime64):\n                sec = int(v.astype(\"datetime64[s]\").astype(\"int64\"))\n                out.append(datetime.fromtimestamp(sec, tz=timezone.utc))\n                continue\n        except Exception:\n            continue\n    return out\n\n\ndef choose_time_index(\n    np: Any,\n    times: Sequence[datetime],\n    issue_dt: datetime,\n    before_min: int,\n    after_min: int,\n) -> Optional[Tuple[int, datetime, float]]:\n    if not times:\n        return None\n    best_idx: Optional[int] = None\n    best_abs_min: Optional[float] = None\n    best_offset_min: Optional[float] = None\n    for i, t in enumerate(times):\n        offset_min = (t - issue_dt).total_seconds() / 60.0\n        if offset_min < -before_min or offset_min > after_min:\n            continue\n        abs_min = abs(offset_min)\n        if best_abs_min is None or abs_min < best_abs_min:\n            best_idx = i\n            best_abs_min = abs_min\n            best_offset_min = offset_min\n    if best_idx is None or best_offset_min is None:\n        return None\n    return best_idx, times[best_idx], best_offset_min\n\n\ndef infer_units_to_kt_factor(units: str) -> float:\n    u = (units or \"\").strip().lower().replace(\" \", \"\")\n    if not u:\n        return 1.94384\n    if \"knot\" in u or \"kt\" in u:\n        return 1.0\n    if \"m/s\" in u or \"ms-1\" in u or \"m*s-1\" in u:\n        return 1.94384\n    return 1.94384\n\n\ndef extract_wind_speed_da(np: Any, xr: Any, ds: Any) -> Tuple[Any, str, str]:\n    candidates_exact = [\n        \"wind_speed\",\n        \"wind_speed_10m\",\n        \"windspeed\",\n        \"windspeed_10m\",\n        \"wind_speed_mean\",\n    ]\n    lower_to_name = {name.lower(): name for name in ds.data_vars}\n    for c in candidates_exact:\n        name = lower_to_name.get(c.lower())\n        if name:\n            da = ds[name]\n            units = str(da.attrs.get(\"units\") or \"\")\n            factor = infer_units_to_kt_factor(units)\n            return da.astype(\"float64\") * factor, name, \"kt\"\n\n    for name in ds.data_vars:\n        lname = name.lower()\n        if \"wind\" in lname and \"speed\" in lname:\n            da = ds[name]\n            units = str(da.attrs.get(\"units\") or \"\")\n            factor = infer_units_to_kt_factor(units)\n            return da.astype(\"float64\") * factor, name, \"kt\"\n\n    u_names = [n for n in ds.data_vars if n.lower() in {\"u10\", \"uwnd\", \"u_wind\", \"eastward_wind\"}]\n    v_names = [n for n in ds.data_vars if n.lower() in {\"v10\", \"vwnd\", \"v_wind\", \"northward_wind\"}]\n    if u_names and v_names:\n        u = ds[u_names[0]].astype(\"float64\")\n        v = ds[v_names[0]].astype(\"float64\")\n        units = str(u.attrs.get(\"units\") or v.attrs.get(\"units\") or \"\")\n        factor = infer_units_to_kt_factor(units)\n        da = np.sqrt(u * u + v * v) * factor\n        da.name = \"wind_speed_from_uv\"\n        return da, \"wind_speed_from_uv\", \"kt\"\n\n    raise RuntimeError(\"no wind-speed variable found in subset dataset\")\n\n\ndef pick_coord_name(da: Any, candidates: Sequence[str]) -> Optional[str]:\n    names = list(da.coords.keys())\n    lower_map = {x.lower(): x for x in names}\n    for cand in candidates:\n        found = lower_map.get(cand.lower())\n        if found:\n            return found\n    for name in names:\n        lname = name.lower()\n        for cand in candidates:\n            if cand.lower() in lname:\n                return name\n    return None\n\n\ndef haversine_km(np: Any, lat1: Any, lon1: Any, lat2: float, lon2: float) -> Any:\n    r = 6371.0\n    lat1_r = np.radians(lat1)\n    lon1_r = np.radians(lon1)\n    lat2_r = math.radians(lat2)\n    lon2_r = math.radians(lon2)\n    dlat = lat1_r - lat2_r\n    dlon = lon1_r - lon2_r\n    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1_r) * math.cos(lat2_r) * np.sin(dlon / 2.0) ** 2\n    return 2.0 * r * np.arcsin(np.sqrt(a))\n\n\ndef infer_cell_area_km2(np: Any, lat_flat: Any, lon_flat: Any, assumed_cell_km: float) -> Any:\n    if lat_flat.size < 4 or lon_flat.size < 4:\n        return np.full(lat_flat.shape, assumed_cell_km * assumed_cell_km, dtype=\"float64\")\n\n    lat_unique = np.unique(np.round(lat_flat, 5))\n    lon_unique = np.unique(np.round(lon_flat, 5))\n    if lat_unique.size < 2 or lon_unique.size < 2:\n        return np.full(lat_flat.shape, assumed_cell_km * assumed_cell_km, dtype=\"float64\")\n\n    dlat = np.median(np.abs(np.diff(lat_unique)))\n    dlon = np.median(np.abs(np.diff(lon_unique)))\n    if not np.isfinite(dlat) or not np.isfinite(dlon) or dlat <= 0 or dlon <= 0:\n        return np.full(lat_flat.shape, assumed_cell_km * assumed_cell_km, dtype=\"float64\")\n\n    dlat_rad = math.radians(float(dlat))\n    dlon_rad = math.radians(float(dlon))\n    r = 6371.0\n    area = (r * r) * dlat_rad * dlon_rad * np.cos(np.radians(lat_flat))\n    area = np.abs(area)\n    floor = max(1.0, assumed_cell_km * assumed_cell_km * 0.2)\n    area = np.where(np.isfinite(area), np.maximum(area, floor), assumed_cell_km * assumed_cell_km)\n    return area\n\n\ndef stat_mean(np: Any, values: Any) -> Optional[float]:\n    if values.size == 0:\n        return None\n    return float(np.nanmean(values))\n\n\ndef stat_max(np: Any, values: Any) -> Optional[float]:\n    if values.size == 0:\n        return None\n    return float(np.nanmax(values))\n\n\ndef stat_p90(np: Any, values: Any) -> Optional[float]:\n    if values.size == 0:\n        return None\n    return float(np.nanpercentile(values, 90))\n\n\ndef round_or_blank(x: Optional[float], digits: int = 4) -> Any:\n    if x is None or not math.isfinite(x):\n        return \"\"\n    return round(x, digits)\n\n\ndef compute_feature_row_from_dataset(\n    np: Any,\n    xr: Any,\n    req: RequestRow,\n    ds: Any,\n    dataset_id: str,\n    args: argparse.Namespace,\n) -> Dict[str, Any]:\n    wind_da, var_name, out_unit = extract_wind_speed_da(np, xr, ds)\n\n    time_name = pick_coord_name(wind_da, [\"time\", \"valid_time\", \"observation_time\"])\n    obs_time_dt = req.issue_dt\n    obs_offset_min = 0.0\n\n    if time_name is not None:\n        times = to_datetime_list(np, wind_da[time_name].values)\n        selected = choose_time_index(\n            np=np,\n            times=times,\n            issue_dt=req.issue_dt,\n            before_min=args.window_before_min,\n            after_min=args.window_after_min,\n        )\n        if selected is None:\n            raise RuntimeError(\"no ASCAT time point in requested issue-time window\")\n        idx, obs_time_dt, obs_offset_min = selected\n        wind_da = wind_da.isel({time_name: idx})\n\n    lat_name = pick_coord_name(wind_da, [\"latitude\", \"lat\"])\n    lon_name = pick_coord_name(wind_da, [\"longitude\", \"lon\"])\n    if not lat_name or not lon_name:\n        raise RuntimeError(\"latitude/longitude coordinates not found in ASCAT subset data\")\n\n    lat_da = wind_da[lat_name]\n    lon_da = wind_da[lon_name]\n    lon_da = ((lon_da + 180) % 360) - 180\n\n    wind_b, lat_b, lon_b = xr.broadcast(wind_da, lat_da, lon_da)\n    wind_arr = np.asarray(wind_b.values, dtype=\"float64\").reshape(-1)\n    lat_arr = np.asarray(lat_b.values, dtype=\"float64\").reshape(-1)\n    lon_arr = np.asarray(lon_b.values, dtype=\"float64\").reshape(-1)\n\n    valid = np.isfinite(wind_arr) & np.isfinite(lat_arr) & np.isfinite(lon_arr)\n    if not valid.any():\n        raise RuntimeError(\"ASCAT subset contains no valid wind cells\")\n\n    wind_arr = wind_arr[valid]\n    lat_arr = lat_arr[valid]\n    lon_arr = lon_arr[valid]\n\n    dist_km = haversine_km(np, lat_arr, lon_arr, req.lat, parse_lon_minus180_180(req.lon))\n    inner_mask = dist_km <= float(args.inner_radius_km)\n    ring_mask = (dist_km > float(args.inner_radius_km)) & (dist_km <= float(args.outer_radius_km))\n    if not inner_mask.any():\n        raise RuntimeError(\"ASCAT subset has no valid inner-radius cells\")\n\n    inner_vals = wind_arr[inner_mask]\n    ring_vals = wind_arr[ring_mask]\n    cell_area = infer_cell_area_km2(np, lat_arr, lon_arr, float(args.assumed_cell_km))\n    inner_area = cell_area[inner_mask]\n\n    ge34_area = float(np.nansum(inner_area[inner_vals >= 34.0])) if inner_vals.size else 0.0\n    ge50_area = float(np.nansum(inner_area[inner_vals >= 50.0])) if inner_vals.size else 0.0\n\n    platform = (\n        str(ds.attrs.get(\"platform\") or \"\")\n        or str(ds.attrs.get(\"platform_name\") or \"\")\n        or str(ds.attrs.get(\"satellite\") or \"\")\n        or dataset_id\n    )\n\n    row = {k: \"\" for k in OUTPUT_FIELDS}\n    for k, v in req.raw.items():\n        if k in row:\n            row[k] = v\n    row[\"request_id\"] = req.request_id\n    row[\"issue_time_utc\"] = req.issue_time_utc\n    row[\"lat\"] = req.lat\n    row[\"lon\"] = req.lon\n    row[\"ascat_status\"] = \"available\"\n    row[\"missing_reason\"] = \"\"\n    row[\"obs_time_utc\"] = obs_time_dt.strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n    row[\"obs_offset_minutes\"] = round(obs_offset_min, 3)\n    row[\"obs_offset_abs_minutes\"] = round(abs(obs_offset_min), 3)\n    row[\"ascat_dataset_id\"] = dataset_id\n    row[\"ascat_platform\"] = platform\n    row[\"ascat_variable\"] = var_name\n    row[\"ascat_units\"] = out_unit\n    row[\"inner_radius_km\"] = args.inner_radius_km\n    row[\"outer_radius_km\"] = args.outer_radius_km\n    row[\"wind_mean_inner_kt\"] = round_or_blank(stat_mean(np, inner_vals))\n    row[\"wind_p90_inner_kt\"] = round_or_blank(stat_p90(np, inner_vals))\n    row[\"wind_max_inner_kt\"] = round_or_blank(stat_max(np, inner_vals))\n    row[\"wind_mean_ring_kt\"] = round_or_blank(stat_mean(np, ring_vals))\n    row[\"wind_p90_ring_kt\"] = round_or_blank(stat_p90(np, ring_vals))\n    row[\"wind_max_ring_kt\"] = round_or_blank(stat_max(np, ring_vals))\n    row[\"wind_area_ge34kt_inner_km2\"] = round_or_blank(ge34_area)\n    row[\"wind_area_ge50kt_inner_km2\"] = round_or_blank(ge50_area)\n    row[\"valid_cell_count\"] = int(inner_vals.size)\n    row[\"qc_has_data\"] = 1\n    row[\"qc_time_within_window\"] = 1\n    return row\n\n\ndef failed_row(req: RequestRow, args: argparse.Namespace, reason: str) -> Dict[str, Any]:\n    out = {k: \"\" for k in OUTPUT_FIELDS}\n    for k, v in req.raw.items():\n        if k in out:\n            out[k] = v\n    out[\"request_id\"] = req.request_id\n    out[\"issue_time_utc\"] = req.issue_time_utc\n    out[\"lat\"] = req.lat\n    out[\"lon\"] = req.lon\n    out[\"ascat_status\"] = \"missing_real_data\"\n    out[\"missing_reason\"] = reason\n    out[\"inner_radius_km\"] = args.inner_radius_km\n    out[\"outer_radius_km\"] = args.outer_radius_km\n    out[\"qc_has_data\"] = 0\n    out[\"qc_time_within_window\"] = 0\n    return out\n\n\ndef process_request(\n    np: Any,\n    xr: Any,\n    copernicusmarine: Any,\n    req: RequestRow,\n    dataset_ids: List[str],\n    args: argparse.Namespace,\n) -> Dict[str, Any]:\n    start_dt = req.issue_dt - timedelta(minutes=args.window_before_min)\n    end_dt = req.issue_dt + timedelta(minutes=args.window_after_min)\n    min_lon, max_lon, min_lat, max_lat = build_bbox(\n        lat=req.lat,\n        lon=req.lon,\n        outer_radius_km=args.outer_radius_km,\n        margin_deg=args.bbox_margin_deg,\n    )\n\n    reasons: List[str] = []\n    for ds_id in dataset_ids:\n        req_token = safe_request_id(req.request_id)\n        stamp = req.issue_dt.strftime(\"%Y%m%dT%H%M\")\n        tmp_file = args.tmp_dir / f\"ascat_{req_token}_{stamp}_{ds_id.replace('/', '_')}.nc\"\n\n        last_err = \"\"\n        for attempt in range(1, max(1, args.max_retries) + 1):\n            try:\n                path = call_subset(\n                    copernicusmarine=copernicusmarine,\n                    dataset_id=ds_id,\n                    out_file=tmp_file,\n                    min_lon=min_lon,\n                    max_lon=max_lon,\n                    min_lat=min_lat,\n                    max_lat=max_lat,\n                    start_dt=start_dt,\n                    end_dt=end_dt,\n                    args=args,\n                )\n                ds = xr.open_dataset(path)\n                try:\n                    row = compute_feature_row_from_dataset(\n                        np=np,\n                        xr=xr,\n                        req=req,\n                        ds=ds,\n                        dataset_id=ds_id,\n                        args=args,\n                    )\n                finally:\n                    ds.close()\n                if not args.keep_temp_files and path.exists():\n                    try:\n                        path.unlink()\n                    except Exception:\n                        pass\n                return row\n            except Exception as exc:\n                last_err = f\"{type(exc).__name__}:{str(exc)[:200]}\"\n                if attempt < args.max_retries:\n                    time.sleep(min(5.0, 0.8 * attempt))\n                continue\n            finally:\n                if not args.keep_temp_files and tmp_file.exists():\n                    try:\n                        tmp_file.unlink()\n                    except Exception:\n                        pass\n\n        reasons.append(f\"{ds_id}=>{last_err or 'unknown_error'}\")\n\n    return failed_row(req, args, \" ; \".join(reasons)[:900] or \"no_ascat_data_or_subset_failed\")\n\n\ndef quantile(values: List[float], p: float) -> Optional[float]:\n    if not values:\n        return None\n    values_sorted = sorted(values)\n    if len(values_sorted) == 1:\n        return values_sorted[0]\n    pos = p * (len(values_sorted) - 1)\n    lo = int(pos)\n    hi = min(lo + 1, len(values_sorted) - 1)\n    w = pos - lo\n    return values_sorted[lo] * (1 - w) + values_sorted[hi] * w\n\n\ndef run(args: argparse.Namespace, rows: List[RequestRow], dataset_ids: List[str]) -> Dict[str, Any]:\n    args.tmp_dir.mkdir(parents=True, exist_ok=True)\n    args.out_csv.parent.mkdir(parents=True, exist_ok=True)\n\n    summary: Dict[str, Any] = {\n        \"generated_at_utc\": datetime.now(timezone.utc).strftime(\"%Y-%m-%dT%H:%M:%SZ\"),\n        \"manifest_csv\": str(args.manifest_csv),\n        \"out_csv\": str(args.out_csv),\n        \"summary_json\": str(args.summary_json),\n        \"dataset_ids_requested\": args.dataset_ids if args.dataset_ids else DEFAULT_DATASET_IDS,\n        \"dataset_ids_used\": dataset_ids,\n        \"requests_total\": len(rows),\n        \"rows_written\": 0,\n        \"available_rows\": 0,\n        \"missing_rows\": 0,\n        \"mean_abs_offset_min_available\": None,\n        \"p50_abs_offset_min_available\": None,\n        \"p90_abs_offset_min_available\": None,\n        \"coverage_by_year\": {},\n    }\n\n    offsets: List[float] = []\n    by_year: Dict[str, Dict[str, int]] = {}\n\n    try:\n        np, xr, copernicusmarine = init_optional_deps()\n    except Exception as exc:\n        print(f\"[WARN] dependency init failed: {type(exc).__name__}: {str(exc)[:260]}\")\n        with args.out_csv.open(\"w\", encoding=\"utf-8\", newline=\"\") as f:\n            writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)\n            writer.writeheader()\n            for req in rows:\n                row = failed_row(req, args, f\"runtime_dependency_error:{type(exc).__name__}\")\n                writer.writerow({k: row.get(k, \"\") for k in OUTPUT_FIELDS})\n                summary[\"rows_written\"] += 1\n                summary[\"missing_rows\"] += 1\n        args.summary_json.parent.mkdir(parents=True, exist_ok=True)\n        args.summary_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n        return summary\n\n    def update_summary(row: Dict[str, Any]) -> None:\n        summary[\"rows_written\"] += 1\n        year = (str(row.get(\"issue_time_utc\") or \"\"))[:4]\n        if year:\n            bucket = by_year.setdefault(year, {\"total\": 0, \"available\": 0, \"missing\": 0})\n            bucket[\"total\"] += 1\n\n        status = (str(row.get(\"ascat_status\") or \"\")).strip()\n        if status == \"available\":\n            summary[\"available_rows\"] += 1\n            if year:\n                by_year[year][\"available\"] += 1\n            off = parse_float(row.get(\"obs_offset_abs_minutes\"))\n            if off is not None:\n                offsets.append(off)\n        else:\n            summary[\"missing_rows\"] += 1\n            if year:\n                by_year[year][\"missing\"] += 1\n\n    with args.out_csv.open(\"w\", encoding=\"utf-8\", newline=\"\") as f:\n        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)\n        writer.writeheader()\n        for i, req in enumerate(rows, start=1):\n            row = process_request(\n                np=np,\n                xr=xr,\n                copernicusmarine=copernicusmarine,\n                req=req,\n                dataset_ids=dataset_ids,\n                args=args,\n            )\n            out_row = {k: row.get(k, \"\") for k in OUTPUT_FIELDS}\n            writer.writerow(out_row)\n            update_summary(out_row)\n            if i % 50 == 0 or i == len(rows):\n                print(f\"processed {i}/{len(rows)}\")\n            time.sleep(max(0.0, args.sleep_sec))\n\n    if offsets:\n        summary[\"mean_abs_offset_min_available\"] = round(sum(offsets) / len(offsets), 3)\n        p50 = quantile(offsets, 0.5)\n        p90 = quantile(offsets, 0.9)\n        summary[\"p50_abs_offset_min_available\"] = round(p50, 3) if p50 is not None else None\n        summary[\"p90_abs_offset_min_available\"] = round(p90, 3) if p90 is not None else None\n\n    coverage_by_year: Dict[str, Dict[str, Any]] = {}\n    for y in sorted(by_year.keys()):\n        total = by_year[y][\"total\"]\n        available = by_year[y][\"available\"]\n        missing = by_year[y][\"missing\"]\n        coverage_by_year[y] = {\n            \"total\": total,\n            \"available\": available,\n            \"missing\": missing,\n            \"coverage_rate\": round((available / total), 6) if total > 0 else 0.0,\n        }\n    summary[\"coverage_by_year\"] = coverage_by_year\n\n    args.summary_json.parent.mkdir(parents=True, exist_ok=True)\n    args.summary_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    return summary\n\n\ndef main() -> int:\n    args = parse_args()\n    dataset_ids = args.dataset_ids if args.dataset_ids else list(DEFAULT_DATASET_IDS)\n    rows = load_manifest(args)\n    if not rows:\n        raise RuntimeError(\"no valid request rows found in manifest\")\n\n    print(\"manifest:\", args.manifest_csv)\n    print(\"rows_to_process:\", len(rows))\n    print(\"dataset_ids_requested:\", dataset_ids)\n    print(\"tmp_dir:\", args.tmp_dir)\n    if args.dry_run:\n        return 0\n\n    summary = run(args=args, rows=rows, dataset_ids=dataset_ids)\n    print(args.out_csv)\n    print(args.summary_json)\n    print(\"rows_written:\", summary[\"rows_written\"])\n    print(\"available_rows:\", summary[\"available_rows\"])\n    print(\"missing_rows:\", summary[\"missing_rows\"])\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
    "extract_recon_features_remote.py": "#!/usr/bin/env python3\n\"\"\"Extract Recon structured features from NHC recon archive text products.\n\nDesign goals:\n1. Remote side downloads/parses raw text products.\n2. Local side stores compact row-wise structured features only.\n3. Fault-tolerant execution for long historical windows.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport re\nimport subprocess\nimport time\nimport urllib.request\nfrom dataclasses import dataclass\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Sequence, Tuple\nfrom urllib.error import HTTPError, URLError\n\n\nDEFAULT_SUBDIRS = [\"REPNT2\", \"REPNT3\", \"AHONT1\", \"AHOPN1\"]\n\nOUTPUT_FIELDS = [\n    \"request_id\",\n    \"storm_id\",\n    \"storm_id_match_status\",\n    \"atcf_storm_id\",\n    \"basin\",\n    \"storm_name\",\n    \"advisory_no\",\n    \"issue_time_utc\",\n    \"lat\",\n    \"lon\",\n    \"source_file\",\n    \"recon_status\",\n    \"missing_reason\",\n    \"recon_obs_time_utc\",\n    \"obs_offset_minutes\",\n    \"obs_offset_abs_minutes\",\n    \"recon_message_type\",\n    \"message_count\",\n    \"recon_source_file\",\n    \"recon_source_subdirs\",\n    \"vdm_min_slp_mb\",\n    \"vdm_max_flight_level_wind_kt\",\n    \"vdm_center_lat\",\n    \"vdm_center_lon\",\n    \"hdob_max_sfmr_wind_kt\",\n    \"hdob_max_flight_level_wind_kt\",\n    \"dropsonde_min_slp_mb\",\n    \"qc_has_data\",\n    \"qc_time_within_window\",\n]\n\nCATALOG_FILE_RE = re.compile(r'href=\"([A-Z0-9_-]+\\.[0-9]{12}\\.txt)\"', re.IGNORECASE)\nVDM_STORM_RE = re.compile(r\"VORTEX DATA MESSAGE\\s+([A-Z]{2}\\d{2}\\d{4})\", re.IGNORECASE)\nVDM_A_RE = re.compile(r\"^\\s*A\\.\\s*(\\d{1,2})/(\\d{2}):(\\d{2}):(\\d{2})Z\", re.IGNORECASE | re.MULTILINE)\nVDM_B_RE = re.compile(\n    r\"^\\s*B\\.\\s*([0-9.]+)\\s*deg\\s*([NS])\\s*([0-9.]+)\\s*deg\\s*([EW])\",\n    re.IGNORECASE | re.MULTILINE,\n)\nVDM_D_RE = re.compile(r\"^\\s*D\\.\\s*(\\d{3,4})\\s*mb\", re.IGNORECASE | re.MULTILINE)\nVDM_J_RE = re.compile(r\"^\\s*J\\.\\s*\\d{3}\\s*deg\\s*(\\d{1,3})\\s*kt\", re.IGNORECASE | re.MULTILINE)\nMAX_FL_WIND_RE = re.compile(r\"MAX FL WIND\\s+(\\d{1,3})\\s*KT\", re.IGNORECASE)\n\nHDOB_HEAD_RE = re.compile(r\"\\bHDOB\\b\", re.IGNORECASE)\nHDOB_LINE_RE = re.compile(r\"^\\s*(\\d{6})\\s+([0-9]{3,5}[NS])\\s+([0-9]{4,6}[EW])\\s+(.+)$\")\nSFMR_RE = re.compile(r\"\\bSFMR[^0-9]{0,10}(\\d{2,3})\\b\", re.IGNORECASE)\nDROPPSONDE_PRESS_RE = re.compile(\n    r\"(?:MIN(?:IMUM)?\\s+)?(?:SFC|SURFACE)?\\s*PRESS(?:URE)?[^0-9]{0,12}(\\d{3,4})\\s*MB\",\n    re.IGNORECASE,\n)\n\n\n@dataclass\nclass RequestRow:\n    request_id: str\n    issue_time_utc: str\n    issue_dt: datetime\n    lat: float\n    lon: float\n    raw: Dict[str, str]\n\n\n@dataclass\nclass CatalogEntry:\n    year: int\n    subdir: str\n    file_name: str\n    file_dt: datetime\n    url: str\n\n\n@dataclass\nclass ParsedReconMessage:\n    source_subdir: str\n    source_file: str\n    source_url: str\n    message_type: str\n    storm_id_extracted: str\n    obs_time_utc: datetime\n    center_lat: Optional[float]\n    center_lon: Optional[float]\n    vdm_min_slp_mb: Optional[float]\n    vdm_max_flight_level_wind_kt: Optional[float]\n    hdob_max_sfmr_wind_kt: Optional[float]\n    hdob_max_flight_level_wind_kt: Optional[float]\n    dropsonde_min_slp_mb: Optional[float]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=\"Extract Recon compact feature table from request manifest via NHC archive.\"\n    )\n    parser.add_argument(\n        \"--manifest-csv\",\n        type=Path,\n        default=Path(\"data/interim/recon/recon_request_manifest.csv\"),\n        help=\"Input request manifest csv.\",\n    )\n    parser.add_argument(\n        \"--out-csv\",\n        type=Path,\n        default=Path(\"data/interim/recon/recon_observation_features.csv\"),\n        help=\"Output feature csv.\",\n    )\n    parser.add_argument(\n        \"--summary-json\",\n        type=Path,\n        default=Path(\"data/interim/recon/recon_observation_features_summary.json\"),\n        help=\"Output summary json.\",\n    )\n    parser.add_argument(\n        \"--base-url\",\n        type=str,\n        default=\"https://www.nhc.noaa.gov/archive/recon\",\n        help=\"NHC recon archive base url.\",\n    )\n    parser.add_argument(\n        \"--subdir\",\n        action=\"append\",\n        dest=\"subdirs\",\n        default=[],\n        help=\"Recon message subdirectory under year folder (repeatable).\",\n    )\n    parser.add_argument(\n        \"--window-before-hours\",\n        type=float,\n        default=12.0,\n        help=\"Look-back window before issue time.\",\n    )\n    parser.add_argument(\n        \"--window-after-hours\",\n        type=float,\n        default=3.0,\n        help=\"Look-forward window after issue time.\",\n    )\n    parser.add_argument(\n        \"--spatial-max-dist-km\",\n        type=float,\n        default=450.0,\n        help=\"When storm-id is unavailable, max center-distance for accepting a message.\",\n    )\n    parser.add_argument(\n        \"--catalog-cache-dir\",\n        type=Path,\n        default=Path(\"/tmp/recon_catalog_cache\"),\n        help=\"Cache directory for listing/message text.\",\n    )\n    parser.add_argument(\n        \"--no-cache\",\n        action=\"store_true\",\n        help=\"Disable local cache for listing and message text.\",\n    )\n    parser.add_argument(\n        \"--max-rows\",\n        type=int,\n        default=0,\n        help=\"If >0, process first N manifest rows.\",\n    )\n    parser.add_argument(\n        \"--only-with-storm-id\",\n        action=\"store_true\",\n        help=\"Keep only rows with non-empty storm_id.\",\n    )\n    parser.add_argument(\n        \"--max-retries\",\n        type=int,\n        default=3,\n        help=\"Retries for HTTP requests.\",\n    )\n    parser.add_argument(\n        \"--sleep-sec\",\n        type=float,\n        default=0.05,\n        help=\"Sleep between manifest rows.\",\n    )\n    parser.add_argument(\n        \"--dry-run\",\n        action=\"store_true\",\n        help=\"Only print resolved config and row counts.\",\n    )\n    return parser.parse_args()\n\n\ndef parse_iso_utc(value: str) -> datetime:\n    txt = (value or \"\").strip()\n    fmts = [\n        \"%Y-%m-%dT%H:%M:%SZ\",\n        \"%Y-%m-%d %H:%M:%S\",\n        \"%Y-%m-%dT%H:%M:%S\",\n        \"%Y-%m-%d %H:%M:%S.%f\",\n        \"%Y-%m-%dT%H:%M:%S.%f\",\n    ]\n    for fmt in fmts:\n        try:\n            return datetime.strptime(txt, fmt).replace(tzinfo=timezone.utc)\n        except ValueError:\n            continue\n    raise ValueError(f\"unsupported datetime format: {value}\")\n\n\ndef parse_float(value: Any) -> Optional[float]:\n    try:\n        if value is None:\n            return None\n        txt = str(value).strip()\n        if not txt:\n            return None\n        return float(txt)\n    except Exception:\n        return None\n\n\ndef norm_lon_to_minus180_180(lon: float) -> float:\n    out = lon\n    while out >= 180:\n        out -= 360\n    while out < -180:\n        out += 360\n    return out\n\n\ndef load_manifest(args: argparse.Namespace) -> List[RequestRow]:\n    if not args.manifest_csv.exists():\n        raise FileNotFoundError(f\"manifest not found: {args.manifest_csv}\")\n    rows: List[RequestRow] = []\n    with args.manifest_csv.open(\"r\", encoding=\"utf-8\", newline=\"\") as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n            issue_time = (row.get(\"issue_time_utc\") or \"\").strip()\n            lat = parse_float(row.get(\"lat\"))\n            lon = parse_float(row.get(\"lon\"))\n            if not issue_time or lat is None or lon is None:\n                continue\n            if args.only_with_storm_id and not (row.get(\"storm_id\") or \"\").strip():\n                continue\n            try:\n                issue_dt = parse_iso_utc(issue_time)\n            except ValueError:\n                continue\n            request_id = (row.get(\"request_id\") or \"\").strip()\n            if not request_id:\n                request_id = f\"REQ_{issue_dt.strftime('%Y%m%dT%H%M%S')}_{len(rows):06d}\"\n            rows.append(\n                RequestRow(\n                    request_id=request_id,\n                    issue_time_utc=issue_dt.strftime(\"%Y-%m-%dT%H:%M:%SZ\"),\n                    issue_dt=issue_dt,\n                    lat=float(lat),\n                    lon=float(lon),\n                    raw={k: (v or \"\").strip() for k, v in row.items()},\n                )\n            )\n    rows.sort(key=lambda x: (x.issue_time_utc, x.request_id))\n    if args.max_rows > 0:\n        rows = rows[: args.max_rows]\n    return rows\n\n\ndef haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:\n    r = 6371.0\n    p1 = math.radians(lat1)\n    p2 = math.radians(lat2)\n    dlat = p2 - p1\n    dlon = math.radians(lon2 - lon1)\n    a = (math.sin(dlat / 2.0) ** 2) + math.cos(p1) * math.cos(p2) * (math.sin(dlon / 2.0) ** 2)\n    return 2.0 * r * math.asin(math.sqrt(a))\n\n\ndef http_get_text(url: str, timeout_sec: int, max_retries: int) -> str:\n    last_exc: Optional[Exception] = None\n    for attempt in range(1, max(1, max_retries) + 1):\n        try:\n            with urllib.request.urlopen(url, timeout=timeout_sec) as resp:\n                return resp.read().decode(\"utf-8\", errors=\"ignore\")\n        except HTTPError as exc:\n            if 400 <= exc.code < 500 and exc.code != 429:\n                detail = \"\"\n                try:\n                    detail = exc.read().decode(\"utf-8\", errors=\"ignore\")\n                except Exception:\n                    detail = \"\"\n                raise RuntimeError(f\"HTTP {exc.code} client error for {url}: {detail[:300]}\") from exc\n            last_exc = exc\n        except URLError as exc:\n            last_exc = exc\n        except Exception as exc:\n            last_exc = exc\n        time.sleep(min(8.0, 0.8 * attempt))\n    if last_exc is not None:\n        # Fallback for environments where Python DNS/network is restricted but curl works.\n        try:\n            proc = subprocess.run(\n                [\"curl\", \"-s\", url],\n                check=True,\n                capture_output=True,\n                text=True,\n                timeout=timeout_sec,\n            )\n            return proc.stdout\n        except Exception:\n            raise last_exc\n    raise RuntimeError(f\"http_get_text failed without exception: {url}\")\n\n\ndef list_catalog_for_subdir(\n    args: argparse.Namespace,\n    year: int,\n    subdir: str,\n) -> List[CatalogEntry]:\n    cache_file = args.catalog_cache_dir / f\"listing_{year}_{subdir}.html\"\n    url = f\"{args.base_url.rstrip('/')}/{year}/{subdir}/\"\n\n    html = \"\"\n    if not args.no_cache and cache_file.exists():\n        html = cache_file.read_text(encoding=\"utf-8\", errors=\"ignore\")\n    else:\n        html = http_get_text(url=url, timeout_sec=60, max_retries=args.max_retries)\n        if not args.no_cache:\n            cache_file.parent.mkdir(parents=True, exist_ok=True)\n            cache_file.write_text(html, encoding=\"utf-8\")\n\n    out: List[CatalogEntry] = []\n    seen = set()\n    for m in CATALOG_FILE_RE.finditer(html):\n        fn = m.group(1).strip()\n        if fn in seen:\n            continue\n        seen.add(fn)\n        ts_match = re.search(r\"\\.(\\d{12})\\.txt$\", fn)\n        if not ts_match:\n            continue\n        try:\n            file_dt = datetime.strptime(ts_match.group(1), \"%Y%m%d%H%M\").replace(tzinfo=timezone.utc)\n        except ValueError:\n            continue\n        out.append(\n            CatalogEntry(\n                year=year,\n                subdir=subdir,\n                file_name=fn,\n                file_dt=file_dt,\n                url=f\"{url}{fn}\",\n            )\n        )\n    out.sort(key=lambda x: x.file_dt)\n    return out\n\n\ndef parse_latlon_token(token: str) -> Optional[float]:\n    txt = (token or \"\").strip().upper()\n    if not txt or txt[-1] not in {\"N\", \"S\", \"E\", \"W\"}:\n        return None\n    hemi = txt[-1]\n    body = txt[:-1]\n    if not body.isdigit():\n        return None\n    if hemi in {\"N\", \"S\"}:\n        if len(body) not in {3, 4}:\n            return None\n    else:\n        if len(body) not in {4, 5}:\n            return None\n    deg = int(body[:-2])\n    minutes = int(body[-2:])\n    value = deg + (minutes / 60.0)\n    if hemi in {\"S\", \"W\"}:\n        value = -value\n    return value\n\n\ndef parse_vdm_message(text: str, file_dt: datetime, source_subdir: str, source_file: str, source_url: str) -> ParsedReconMessage:\n    storm_id = \"\"\n    m = VDM_STORM_RE.search(text)\n    if m:\n        storm_id = m.group(1).upper()\n\n    obs_time = file_dt\n    ma = VDM_A_RE.search(text)\n    if ma:\n        day = int(ma.group(1))\n        hh = int(ma.group(2))\n        mm = int(ma.group(3))\n        ss = int(ma.group(4))\n        try:\n            obs_time = file_dt.replace(day=day, hour=hh, minute=mm, second=ss, microsecond=0)\n        except ValueError:\n            obs_time = file_dt.replace(hour=hh, minute=mm, second=ss, microsecond=0)\n\n    center_lat = None\n    center_lon = None\n    mb = VDM_B_RE.search(text)\n    if mb:\n        lat = float(mb.group(1)) * (1.0 if mb.group(2).upper() == \"N\" else -1.0)\n        lon = float(mb.group(3)) * (-1.0 if mb.group(4).upper() == \"W\" else 1.0)\n        center_lat = lat\n        center_lon = norm_lon_to_minus180_180(lon)\n\n    min_slp = None\n    md = VDM_D_RE.search(text)\n    if md:\n        min_slp = parse_float(md.group(1))\n\n    max_fl = None\n    mj = VDM_J_RE.search(text)\n    if mj:\n        max_fl = parse_float(mj.group(1))\n    else:\n        mmf = MAX_FL_WIND_RE.search(text)\n        if mmf:\n            max_fl = parse_float(mmf.group(1))\n\n    return ParsedReconMessage(\n        source_subdir=source_subdir,\n        source_file=source_file,\n        source_url=source_url,\n        message_type=\"VDM\",\n        storm_id_extracted=storm_id,\n        obs_time_utc=obs_time,\n        center_lat=center_lat,\n        center_lon=center_lon,\n        vdm_min_slp_mb=min_slp,\n        vdm_max_flight_level_wind_kt=max_fl,\n        hdob_max_sfmr_wind_kt=None,\n        hdob_max_flight_level_wind_kt=None,\n        dropsonde_min_slp_mb=None,\n    )\n\n\ndef parse_hdob_message(text: str, file_dt: datetime, source_subdir: str, source_file: str, source_url: str) -> ParsedReconMessage:\n    center_lat = None\n    center_lon = None\n    max_fl_wind = None\n    max_sfmr = None\n\n    sfmr_vals = [parse_float(x) for x in SFMR_RE.findall(text)]\n    sfmr_vals = [x for x in sfmr_vals if x is not None]\n    if sfmr_vals:\n        max_sfmr = max(sfmr_vals)\n\n    for ln in text.splitlines():\n        m = HDOB_LINE_RE.match(ln)\n        if not m:\n            continue\n        lat_t = parse_latlon_token(m.group(2))\n        lon_t = parse_latlon_token(m.group(3))\n        if lat_t is not None and lon_t is not None and center_lat is None:\n            center_lat = lat_t\n            center_lon = norm_lon_to_minus180_180(lon_t)\n\n        rest_tokens = m.group(4).split()\n        speed_candidates: List[float] = []\n        for tok in rest_tokens:\n            if tok.isdigit():\n                if len(tok) == 5:\n                    spd = parse_float(tok[-2:])\n                    if spd is not None and 0 <= spd <= 150:\n                        speed_candidates.append(spd)\n                elif len(tok) == 6:\n                    spd = parse_float(tok[-3:])\n                    if spd is not None and 0 <= spd <= 220:\n                        speed_candidates.append(spd)\n        if speed_candidates:\n            cur = max(speed_candidates)\n            max_fl_wind = cur if max_fl_wind is None else max(max_fl_wind, cur)\n\n    mmf = MAX_FL_WIND_RE.search(text)\n    if mmf:\n        v = parse_float(mmf.group(1))\n        if v is not None:\n            max_fl_wind = v if max_fl_wind is None else max(max_fl_wind, v)\n\n    return ParsedReconMessage(\n        source_subdir=source_subdir,\n        source_file=source_file,\n        source_url=source_url,\n        message_type=\"HDOB\",\n        storm_id_extracted=\"\",\n        obs_time_utc=file_dt,\n        center_lat=center_lat,\n        center_lon=center_lon,\n        vdm_min_slp_mb=None,\n        vdm_max_flight_level_wind_kt=None,\n        hdob_max_sfmr_wind_kt=max_sfmr,\n        hdob_max_flight_level_wind_kt=max_fl_wind,\n        dropsonde_min_slp_mb=None,\n    )\n\n\ndef parse_dropsonde_message(text: str, file_dt: datetime, source_subdir: str, source_file: str, source_url: str) -> ParsedReconMessage:\n    press_vals = [parse_float(x) for x in DROPPSONDE_PRESS_RE.findall(text)]\n    press_vals = [x for x in press_vals if x is not None]\n    min_press = min(press_vals) if press_vals else None\n\n    return ParsedReconMessage(\n        source_subdir=source_subdir,\n        source_file=source_file,\n        source_url=source_url,\n        message_type=\"DROPSONDE\",\n        storm_id_extracted=\"\",\n        obs_time_utc=file_dt,\n        center_lat=None,\n        center_lon=None,\n        vdm_min_slp_mb=None,\n        vdm_max_flight_level_wind_kt=None,\n        hdob_max_sfmr_wind_kt=None,\n        hdob_max_flight_level_wind_kt=None,\n        dropsonde_min_slp_mb=min_press,\n    )\n\n\ndef parse_message(text: str, entry: CatalogEntry) -> ParsedReconMessage:\n    up = text.upper()\n    if \"VORTEX DATA MESSAGE\" in up:\n        return parse_vdm_message(\n            text=text,\n            file_dt=entry.file_dt,\n            source_subdir=entry.subdir,\n            source_file=entry.file_name,\n            source_url=entry.url,\n        )\n    if HDOB_HEAD_RE.search(text):\n        return parse_hdob_message(\n            text=text,\n            file_dt=entry.file_dt,\n            source_subdir=entry.subdir,\n            source_file=entry.file_name,\n            source_url=entry.url,\n        )\n    if \"XXAA\" in up or \"XXBB\" in up or \"UZNT13\" in up:\n        return parse_dropsonde_message(\n            text=text,\n            file_dt=entry.file_dt,\n            source_subdir=entry.subdir,\n            source_file=entry.file_name,\n            source_url=entry.url,\n        )\n    return ParsedReconMessage(\n        source_subdir=entry.subdir,\n        source_file=entry.file_name,\n        source_url=entry.url,\n        message_type=entry.subdir,\n        storm_id_extracted=\"\",\n        obs_time_utc=entry.file_dt,\n        center_lat=None,\n        center_lon=None,\n        vdm_min_slp_mb=None,\n        vdm_max_flight_level_wind_kt=None,\n        hdob_max_sfmr_wind_kt=None,\n        hdob_max_flight_level_wind_kt=None,\n        dropsonde_min_slp_mb=None,\n    )\n\n\ndef fetch_message_text(args: argparse.Namespace, entry: CatalogEntry) -> str:\n    cache_file = args.catalog_cache_dir / \"messages\" / str(entry.year) / entry.subdir / entry.file_name\n    if not args.no_cache and cache_file.exists():\n        return cache_file.read_text(encoding=\"utf-8\", errors=\"ignore\")\n    text = http_get_text(url=entry.url, timeout_sec=60, max_retries=args.max_retries)\n    if not args.no_cache:\n        cache_file.parent.mkdir(parents=True, exist_ok=True)\n        cache_file.write_text(text, encoding=\"utf-8\")\n    return text\n\n\ndef build_catalog(args: argparse.Namespace, rows: Sequence[RequestRow], subdirs: List[str]) -> List[CatalogEntry]:\n    years = set()\n    before = timedelta(hours=args.window_before_hours)\n    after = timedelta(hours=args.window_after_hours)\n    for r in rows:\n        years.add((r.issue_dt - before).year)\n        years.add((r.issue_dt + after).year)\n\n    catalog: List[CatalogEntry] = []\n    for year in sorted(years):\n        for subdir in subdirs:\n            try:\n                entries = list_catalog_for_subdir(args=args, year=year, subdir=subdir)\n            except Exception as exc:\n                print(f\"[WARN] catalog fetch failed for {year}/{subdir}: {type(exc).__name__}: {str(exc)[:220]}\")\n                continue\n            catalog.extend(entries)\n            print(f\"catalog {year}/{subdir}: {len(entries)} files\")\n    catalog.sort(key=lambda x: x.file_dt)\n    return catalog\n\n\ndef pick_candidates(req: RequestRow, catalog: Sequence[CatalogEntry], args: argparse.Namespace) -> List[CatalogEntry]:\n    start_dt = req.issue_dt - timedelta(hours=args.window_before_hours)\n    end_dt = req.issue_dt + timedelta(hours=args.window_after_hours)\n    return [e for e in catalog if start_dt <= e.file_dt <= end_dt]\n\n\ndef is_message_match(req: RequestRow, msg: ParsedReconMessage, args: argparse.Namespace) -> bool:\n    atcf_id = (req.raw.get(\"atcf_storm_id\") or \"\").strip().upper()\n    if msg.storm_id_extracted and atcf_id and msg.storm_id_extracted != atcf_id:\n        return False\n\n    if msg.center_lat is not None and msg.center_lon is not None and not (msg.storm_id_extracted and atcf_id):\n        d = haversine_km(req.lat, norm_lon_to_minus180_180(req.lon), msg.center_lat, msg.center_lon)\n        if d > args.spatial_max_dist_km:\n            return False\n\n    return True\n\n\ndef aggregate_messages(req: RequestRow, msgs: List[ParsedReconMessage], args: argparse.Namespace) -> Dict[str, Any]:\n    if not msgs:\n        out = {k: \"\" for k in OUTPUT_FIELDS}\n        for k, v in req.raw.items():\n            if k in out:\n                out[k] = v\n        out[\"request_id\"] = req.request_id\n        out[\"issue_time_utc\"] = req.issue_time_utc\n        out[\"lat\"] = req.lat\n        out[\"lon\"] = req.lon\n        out[\"recon_status\"] = \"missing_real_data\"\n        out[\"missing_reason\"] = \"no_recon_message_matched_time_and_filter\"\n        out[\"qc_has_data\"] = 0\n        out[\"qc_time_within_window\"] = 0\n        return out\n\n    msgs_sorted = sorted(msgs, key=lambda x: abs((x.obs_time_utc - req.issue_dt).total_seconds()))\n    nearest = msgs_sorted[0]\n    offset_min = (nearest.obs_time_utc - req.issue_dt).total_seconds() / 60.0\n\n    def min_opt(values: Sequence[Optional[float]]) -> Optional[float]:\n        arr = [x for x in values if x is not None]\n        return min(arr) if arr else None\n\n    def max_opt(values: Sequence[Optional[float]]) -> Optional[float]:\n        arr = [x for x in values if x is not None]\n        return max(arr) if arr else None\n\n    vdm_msgs = [m for m in msgs if m.message_type == \"VDM\"]\n    nearest_vdm = None\n    if vdm_msgs:\n        nearest_vdm = sorted(vdm_msgs, key=lambda x: abs((x.obs_time_utc - req.issue_dt).total_seconds()))[0]\n\n    out = {k: \"\" for k in OUTPUT_FIELDS}\n    for k, v in req.raw.items():\n        if k in out:\n            out[k] = v\n    out[\"request_id\"] = req.request_id\n    out[\"issue_time_utc\"] = req.issue_time_utc\n    out[\"lat\"] = req.lat\n    out[\"lon\"] = req.lon\n    out[\"recon_status\"] = \"available\"\n    out[\"missing_reason\"] = \"\"\n    out[\"recon_obs_time_utc\"] = nearest.obs_time_utc.strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n    out[\"obs_offset_minutes\"] = round(offset_min, 3)\n    out[\"obs_offset_abs_minutes\"] = round(abs(offset_min), 3)\n    out[\"recon_message_type\"] = \",\".join(sorted({m.message_type for m in msgs}))\n    out[\"message_count\"] = len(msgs)\n    out[\"recon_source_file\"] = nearest.source_url\n    out[\"recon_source_subdirs\"] = \",\".join(sorted({m.source_subdir for m in msgs}))\n    out[\"vdm_min_slp_mb\"] = min_opt([m.vdm_min_slp_mb for m in msgs]) or \"\"\n    out[\"vdm_max_flight_level_wind_kt\"] = max_opt([m.vdm_max_flight_level_wind_kt for m in msgs]) or \"\"\n    out[\"vdm_center_lat\"] = nearest_vdm.center_lat if nearest_vdm and nearest_vdm.center_lat is not None else \"\"\n    out[\"vdm_center_lon\"] = nearest_vdm.center_lon if nearest_vdm and nearest_vdm.center_lon is not None else \"\"\n    out[\"hdob_max_sfmr_wind_kt\"] = max_opt([m.hdob_max_sfmr_wind_kt for m in msgs]) or \"\"\n    out[\"hdob_max_flight_level_wind_kt\"] = max_opt([m.hdob_max_flight_level_wind_kt for m in msgs]) or \"\"\n    out[\"dropsonde_min_slp_mb\"] = min_opt([m.dropsonde_min_slp_mb for m in msgs]) or \"\"\n    out[\"qc_has_data\"] = 1\n    out[\"qc_time_within_window\"] = 1\n    return out\n\n\ndef quantile(values: List[float], p: float) -> Optional[float]:\n    if not values:\n        return None\n    values_sorted = sorted(values)\n    if len(values_sorted) == 1:\n        return values_sorted[0]\n    pos = p * (len(values_sorted) - 1)\n    lo = int(pos)\n    hi = min(lo + 1, len(values_sorted) - 1)\n    w = pos - lo\n    return values_sorted[lo] * (1 - w) + values_sorted[hi] * w\n\n\ndef run(args: argparse.Namespace, rows: List[RequestRow], catalog: List[CatalogEntry], subdirs: List[str]) -> Dict[str, Any]:\n    args.out_csv.parent.mkdir(parents=True, exist_ok=True)\n    args.summary_json.parent.mkdir(parents=True, exist_ok=True)\n\n    summary: Dict[str, Any] = {\n        \"generated_at_utc\": datetime.now(timezone.utc).strftime(\"%Y-%m-%dT%H:%M:%SZ\"),\n        \"manifest_csv\": str(args.manifest_csv),\n        \"out_csv\": str(args.out_csv),\n        \"summary_json\": str(args.summary_json),\n        \"base_url\": args.base_url,\n        \"subdirs_used\": subdirs,\n        \"catalog_entries_total\": len(catalog),\n        \"requests_total\": len(rows),\n        \"rows_written\": 0,\n        \"available_rows\": 0,\n        \"missing_rows\": 0,\n        \"mean_abs_offset_min_available\": None,\n        \"p50_abs_offset_min_available\": None,\n        \"p90_abs_offset_min_available\": None,\n        \"coverage_by_year\": {},\n    }\n\n    offsets: List[float] = []\n    by_year: Dict[str, Dict[str, int]] = {}\n\n    def update_summary(row: Dict[str, Any]) -> None:\n        summary[\"rows_written\"] += 1\n        year = (str(row.get(\"issue_time_utc\") or \"\"))[:4]\n        if year:\n            bucket = by_year.setdefault(year, {\"total\": 0, \"available\": 0, \"missing\": 0})\n            bucket[\"total\"] += 1\n\n        status = (str(row.get(\"recon_status\") or \"\")).strip()\n        if status == \"available\":\n            summary[\"available_rows\"] += 1\n            if year:\n                by_year[year][\"available\"] += 1\n            off = parse_float(row.get(\"obs_offset_abs_minutes\"))\n            if off is not None:\n                offsets.append(off)\n        else:\n            summary[\"missing_rows\"] += 1\n            if year:\n                by_year[year][\"missing\"] += 1\n\n    with args.out_csv.open(\"w\", encoding=\"utf-8\", newline=\"\") as f:\n        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)\n        writer.writeheader()\n\n        for i, req in enumerate(rows, start=1):\n            candidates = pick_candidates(req=req, catalog=catalog, args=args)\n            parsed_msgs: List[ParsedReconMessage] = []\n\n            for entry in candidates:\n                try:\n                    text = fetch_message_text(args=args, entry=entry)\n                    msg = parse_message(text=text, entry=entry)\n                except Exception:\n                    continue\n                if is_message_match(req=req, msg=msg, args=args):\n                    parsed_msgs.append(msg)\n\n            row = aggregate_messages(req=req, msgs=parsed_msgs, args=args)\n            out_row = {k: row.get(k, \"\") for k in OUTPUT_FIELDS}\n            writer.writerow(out_row)\n            update_summary(out_row)\n\n            if i % 50 == 0 or i == len(rows):\n                print(f\"processed {i}/{len(rows)}\")\n            time.sleep(max(0.0, args.sleep_sec))\n\n    if offsets:\n        summary[\"mean_abs_offset_min_available\"] = round(sum(offsets) / len(offsets), 3)\n        p50 = quantile(offsets, 0.5)\n        p90 = quantile(offsets, 0.9)\n        summary[\"p50_abs_offset_min_available\"] = round(p50, 3) if p50 is not None else None\n        summary[\"p90_abs_offset_min_available\"] = round(p90, 3) if p90 is not None else None\n\n    coverage_by_year: Dict[str, Dict[str, Any]] = {}\n    for y in sorted(by_year.keys()):\n        total = by_year[y][\"total\"]\n        available = by_year[y][\"available\"]\n        missing = by_year[y][\"missing\"]\n        coverage_by_year[y] = {\n            \"total\": total,\n            \"available\": available,\n            \"missing\": missing,\n            \"coverage_rate\": round((available / total), 6) if total > 0 else 0.0,\n        }\n    summary[\"coverage_by_year\"] = coverage_by_year\n\n    args.summary_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    return summary\n\n\ndef main() -> int:\n    args = parse_args()\n    subdirs = args.subdirs if args.subdirs else list(DEFAULT_SUBDIRS)\n    rows = load_manifest(args)\n    if not rows:\n        raise RuntimeError(\"no valid request rows found in manifest\")\n\n    print(\"manifest:\", args.manifest_csv)\n    print(\"rows_to_process:\", len(rows))\n    print(\"base_url:\", args.base_url)\n    print(\"subdirs:\", subdirs)\n    if args.dry_run:\n        return 0\n\n    if not args.no_cache:\n        args.catalog_cache_dir.mkdir(parents=True, exist_ok=True)\n    catalog = build_catalog(args=args, rows=rows, subdirs=subdirs)\n    summary = run(args=args, rows=rows, catalog=catalog, subdirs=subdirs)\n\n    print(args.out_csv)\n    print(args.summary_json)\n    print(\"rows_written:\", summary[\"rows_written\"])\n    print(\"available_rows:\", summary[\"available_rows\"])\n    print(\"missing_rows:\", summary[\"missing_rows\"])\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
}

for name, content in embedded_scripts.items():
    fp = SCRIPTS_DIR / name
    fp.write_text(content, encoding="utf-8")
    print("wrote", fp)

In [ ]:
def load_csv_rows(path: Path):
    with path.open("r", encoding="utf-8", newline="") as f:
        r = csv.DictReader(f)
        return list(r), (r.fieldnames or [])


def write_csv(path: Path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def split_manifest_by_year(src_csv: Path, out_dir: Path, year_start: int, year_end: int):
    rows, fieldnames = load_csv_rows(src_csv)
    by_year = {}
    for row in rows:
        y_txt = str(row.get("issue_time_utc") or "")[:4]
        if not y_txt.isdigit():
            continue
        y = int(y_txt)
        if y < year_start or y > year_end:
            continue
        by_year.setdefault(y, []).append(row)

    out = {}
    out_dir.mkdir(parents=True, exist_ok=True)
    for y in range(year_start, year_end + 1):
        yr_rows = by_year.get(y, [])
        fp = out_dir / f"manifest_{y}.csv"
        write_csv(fp, yr_rows, fieldnames)
        out[y] = fp
        print(f"split {src_csv.name}: year={y}, rows={len(yr_rows)}")
    return out


def _parse_float(x):
    try:
        if x is None:
            return None
        t = str(x).strip()
        if not t:
            return None
        return float(t)
    except Exception:
        return None


def _quantile(values, p):
    if not values:
        return None
    arr = sorted(values)
    if len(arr) == 1:
        return arr[0]
    pos = p * (len(arr) - 1)
    lo = int(pos)
    hi = min(lo + 1, len(arr) - 1)
    w = pos - lo
    return arr[lo] * (1 - w) + arr[hi] * w


def _cleanup_ascat_tmp(tmp_dir: Path):
    if not tmp_dir.exists():
        return
    removed = 0
    for fp in tmp_dir.glob("ascat_*.nc"):
        try:
            fp.unlink()
            removed += 1
        except Exception:
            pass
    if removed:
        print(f"[CLEANUP] removed {removed} ASCAT temp files from {tmp_dir}")


def _build_chunk_manifests(rows, fieldnames, out_dir: Path, year: int, chunk_rows: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    out = []
    if chunk_rows <= 0:
        chunk_rows = len(rows)
    total = len(rows)
    idx = 0
    chunk_id = 1
    while idx < total:
        sub = rows[idx : idx + chunk_rows]
        fp = out_dir / f"manifest_{year}_chunk_{chunk_id:04d}.csv"
        write_csv(fp, sub, fieldnames)
        out.append(fp)
        idx += chunk_rows
        chunk_id += 1
    return out


def _merge_chunk_outputs_for_year(kind: str, status_col: str, year: int, chunk_feature_files, chunk_summary_files, out_csv: Path, out_json: Path):
    all_rows = []
    fieldnames = None
    offsets = []

    for fp in chunk_feature_files:
        if not fp.exists():
            continue
        rows, fns = load_csv_rows(fp)
        if fieldnames is None:
            fieldnames = fns
        all_rows.extend(rows)
        for r in rows:
            if str(r.get(status_col) or "").strip() == "available":
                off = _parse_float(r.get("obs_offset_abs_minutes"))
                if off is not None:
                    offsets.append(off)

    if fieldnames is None:
        raise RuntimeError(f"no chunk feature outputs found for {kind} year={year}")

    write_csv(out_csv, all_rows, fieldnames)
    available_rows = sum(1 for r in all_rows if str(r.get(status_col) or "").strip() == "available")
    missing_rows = len(all_rows) - available_rows

    summary = {
        "generated_from": "single_notebook_yearly_chunks",
        "year": year,
        "rows_written": len(all_rows),
        "available_rows": available_rows,
        "missing_rows": missing_rows,
        "mean_abs_offset_min_available": round(sum(offsets) / len(offsets), 3) if offsets else None,
        "p50_abs_offset_min_available": round(_quantile(offsets, 0.5), 3) if offsets else None,
        "p90_abs_offset_min_available": round(_quantile(offsets, 0.9), 3) if offsets else None,
        "chunk_feature_files": [str(x) for x in chunk_feature_files],
        "chunk_summary_files": [str(x) for x in chunk_summary_files if x.exists()],
    }

    # Carry dataset_ids_used for ASCAT when available.
    dataset_ids_used = []
    for sp in chunk_summary_files:
        if not sp.exists():
            continue
        try:
            obj = json.loads(sp.read_text(encoding="utf-8"))
            for ds in obj.get("dataset_ids_used", []):
                if ds not in dataset_ids_used:
                    dataset_ids_used.append(ds)
        except Exception:
            pass
    if dataset_ids_used:
        summary["dataset_ids_used"] = dataset_ids_used

    out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")


def merge_yearly_outputs(kind: str, status_col: str, year_start: int, year_end: int, feature_dir: Path, summary_dir: Path, final_csv: Path, final_summary: Path):
    all_rows = []
    fieldnames = None
    by_year = {}
    summary_files = []
    dataset_ids_used = []

    for y in range(year_start, year_end + 1):
        fp = feature_dir / f"{kind}_observation_features_{y}.csv"
        sp = summary_dir / f"{kind}_observation_features_{y}_summary.json"
        if sp.exists():
            summary_files.append(str(sp))
            try:
                obj = json.loads(sp.read_text(encoding="utf-8"))
                for ds in obj.get("dataset_ids_used", []):
                    if ds not in dataset_ids_used:
                        dataset_ids_used.append(ds)
            except Exception:
                pass
        if not fp.exists():
            continue
        rows, fns = load_csv_rows(fp)
        if fieldnames is None:
            fieldnames = fns
        all_rows.extend(rows)
        avail = sum(1 for r in rows if str(r.get(status_col) or "").strip() == "available")
        miss = len(rows) - avail
        by_year[str(y)] = {
            "total": len(rows),
            "available": avail,
            "missing": miss,
            "coverage_rate": round((avail / len(rows)), 6) if rows else 0.0,
        }

    if fieldnames is None:
        raise RuntimeError(f"no yearly outputs found for {kind}")

    write_csv(final_csv, all_rows, fieldnames)
    available_rows = sum(1 for r in all_rows if str(r.get(status_col) or "").strip() == "available")
    missing_rows = len(all_rows) - available_rows
    out_summary = {
        "generated_from": "single_notebook_yearly_controlled_runs",
        "requests_total": len(all_rows),
        "rows_written": len(all_rows),
        "available_rows": available_rows,
        "missing_rows": missing_rows,
        "coverage_by_year": by_year,
        "yearly_summary_files": summary_files,
    }
    if dataset_ids_used:
        out_summary["dataset_ids_used"] = dataset_ids_used

    final_summary.write_text(json.dumps(out_summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("merged:", final_csv)
    print("summary:", final_summary)


def run_yearly(
    kind: str,
    extractor_script: Path,
    src_manifest: Path,
    out_root: Path,
    year_start: int,
    year_end: int,
    only_with_storm_id: bool,
    max_rows: int,
    max_retries: int,
    sleep_sec: float,
    extra_args=None,
    chunk_rows=0,
    cleanup_chunk_intermediate=True,
    cleanup_ascat_tmp_dir=True,
    ascat_tmp_dir="/tmp/ascat_subset_tmp",
):
    status_col = "ascat_status" if kind == "ascat" else "recon_status"
    manifests_dir = out_root / "full_by_year" / "manifests"
    features_dir = out_root / "full_by_year" / "features"
    summaries_dir = out_root / "full_by_year" / "summaries"
    chunk_root = out_root / "full_by_year" / "chunk_work"
    manifests_dir.mkdir(parents=True, exist_ok=True)
    features_dir.mkdir(parents=True, exist_ok=True)
    summaries_dir.mkdir(parents=True, exist_ok=True)
    chunk_root.mkdir(parents=True, exist_ok=True)

    by_year = split_manifest_by_year(src_manifest, manifests_dir, year_start, year_end)

    for y in range(year_start, year_end + 1):
        mf = by_year[y]
        out_csv = features_dir / f"{kind}_observation_features_{y}.csv"
        out_json = summaries_dir / f"{kind}_observation_features_{y}_summary.json"

        rows, fieldnames = load_csv_rows(mf)
        if max_rows > 0:
            rows = rows[:max_rows]
        if not rows:
            print(f"[SKIP] {kind} year={y}: empty manifest")
            continue

        if out_csv.exists() and out_json.exists():
            print(f"[SKIP] {kind} year={y}: already done")
            continue

        chunk_manifest_dir = chunk_root / f"{kind}_{y}" / "manifests"
        chunk_feature_dir = chunk_root / f"{kind}_{y}" / "features"
        chunk_summary_dir = chunk_root / f"{kind}_{y}" / "summaries"
        chunk_feature_dir.mkdir(parents=True, exist_ok=True)
        chunk_summary_dir.mkdir(parents=True, exist_ok=True)

        chunk_manifest_files = _build_chunk_manifests(
            rows=rows,
            fieldnames=fieldnames,
            out_dir=chunk_manifest_dir,
            year=y,
            chunk_rows=chunk_rows,
        )
        print(f"[{kind}] year={y} total_rows={len(rows)} chunk_rows={chunk_rows or len(rows)} chunks={len(chunk_manifest_files)}")

        chunk_feature_files = []
        chunk_summary_files = []

        for idx, cmf in enumerate(chunk_manifest_files, start=1):
            c_out_csv = chunk_feature_dir / f"{kind}_observation_features_{y}_chunk_{idx:04d}.csv"
            c_out_json = chunk_summary_dir / f"{kind}_observation_features_{y}_chunk_{idx:04d}_summary.json"
            chunk_feature_files.append(c_out_csv)
            chunk_summary_files.append(c_out_json)

            if c_out_csv.exists() and c_out_json.exists():
                print(f"[SKIP] {kind} year={y} chunk={idx}: already done")
                continue

            cmd = [
                "python3", str(extractor_script),
                "--manifest-csv", str(cmf),
                "--out-csv", str(c_out_csv),
                "--summary-json", str(c_out_json),
                "--max-retries", str(max_retries),
                "--sleep-sec", str(sleep_sec),
            ]
            if only_with_storm_id:
                cmd.append("--only-with-storm-id")
            if extra_args:
                cmd.extend(list(extra_args))

            run(cmd)

            # Force cleanup between chunks for constrained CDS pods.
            if kind == "ascat" and cleanup_ascat_tmp_dir:
                _cleanup_ascat_tmp(Path(ascat_tmp_dir))

        _merge_chunk_outputs_for_year(
            kind=kind,
            status_col=status_col,
            year=y,
            chunk_feature_files=chunk_feature_files,
            chunk_summary_files=chunk_summary_files,
            out_csv=out_csv,
            out_json=out_json,
        )

        if cleanup_chunk_intermediate:
            # Keep only yearly outputs after successful merge.
            for fp in chunk_manifest_files + chunk_feature_files + chunk_summary_files:
                try:
                    if fp.exists():
                        fp.unlink()
                except Exception:
                    pass

    final_csv = out_root / f"{kind}_observation_features_full.csv"
    final_summary = out_root / f"{kind}_observation_features_full_summary.json"
    merge_yearly_outputs(
        kind=kind,
        status_col=status_col,
        year_start=year_start,
        year_end=year_end,
        feature_dir=features_dir,
        summary_dir=summaries_dir,
        final_csv=final_csv,
        final_summary=final_summary,
    )
    return final_csv, final_summary


In [ ]:
# Step 1: resolve manifests

# Bootstrap guards: allow this cell to fail with actionable message instead of NameError.
if "CONFIG" not in globals():
    CONFIG = {
        "workdir": "/home/jovyan/mystorage/cds_obs_pipeline_runtime",
        "build_manifest_in_notebook": False,
        "noaa_root": "/home/jovyan/mystorage/noaa",
        "crosswalk_csv": "/home/jovyan/mystorage/storm_id_crosswalk.csv",
        "year_start": 2016,
        "year_end": 2016,
        "ascat_manifest_input": "/home/jovyan/mystorage/ascat_request_manifest.csv",
        "recon_manifest_input": "/home/jovyan/mystorage/recon_request_manifest.csv",
        "ascat_username": "",
        "ascat_password": "",
        "ascat_credentials_file": "",
        "ascat_service": "",
        "allow_interactive_credential_prompt": False,
        "run_ascat": True,
        "run_recon": True,
        "only_with_storm_id": True,
        "ascat_max_rows": 0,
        "recon_max_rows": 0,
        "ascat_max_retries": 3,
        "recon_max_retries": 3,
        "ascat_sleep_sec": 0.2,
    "recon_sleep_sec": 0.1,
    "ascat_chunk_rows": 5,
    "recon_chunk_rows": 50,
    "cleanup_chunk_intermediate": True,
    "cleanup_ascat_tmp_dir": True,
    "ascat_tmp_dir": "/tmp/ascat_subset_tmp",
        "safe_mode_note": "run one year per session on 4CPU/16GB pod",
        "ascat_window_before_min": 120,
        "ascat_window_after_min": 120,
        "ascat_outer_radius_km": 350.0,
        "ascat_bbox_margin_deg": 0.2,
}
    print("[WARN] CONFIG was missing. Default CONFIG has been injected.")

# Always-applied runtime override.
# Fill these values if you want to force override, even when CONFIG already exists.
CONFIG_RUNTIME_OVERRIDE = {
    "ascat_username": "",
    "ascat_password": "",
    "ascat_credentials_file": "",
    "ascat_service": "",
    # Optional alias keys also supported:
    # "username": "...",
    # "password": "...",
}
for k, v in CONFIG_RUNTIME_OVERRIDE.items():
    if isinstance(v, str):
        if v.strip():
            CONFIG[k] = v.strip()
    else:
        CONFIG[k] = v

if "Path" not in globals():
    from pathlib import Path

if "run" not in globals():
    import subprocess

    def run(cmd, check=True):
        print("\n[RUN]", " ".join(cmd))
        proc = subprocess.run(cmd, text=True)
        if check and proc.returncode != 0:
            raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd)}")

if "WORKDIR" not in globals():
    WORKDIR = Path(CONFIG["workdir"]).resolve()
if "SCRIPTS_DIR" not in globals():
    SCRIPTS_DIR = WORKDIR / "scripts"
if "MANIFEST_DIR" not in globals():
    MANIFEST_DIR = WORKDIR / "manifests"
if "ASCAT_DIR" not in globals():
    ASCAT_DIR = WORKDIR / "ascat"
if "RECON_DIR" not in globals():
    RECON_DIR = WORKDIR / "recon"

for p in [WORKDIR, SCRIPTS_DIR, MANIFEST_DIR, ASCAT_DIR, RECON_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Ensure extractor scripts are available.
required_scripts = [
    SCRIPTS_DIR / "build_obs_request_manifest.py",
    SCRIPTS_DIR / "extract_ascat_features_remote.py",
    SCRIPTS_DIR / "extract_recon_features_remote.py",
]
missing_scripts = [str(x) for x in required_scripts if not x.exists()]
if missing_scripts:
    if "embedded_scripts" in globals():
        for name, content in embedded_scripts.items():
            fp = SCRIPTS_DIR / name
            fp.write_text(content, encoding="utf-8")
            print("wrote", fp)
    else:
        raise RuntimeError(
            "embedded extractor scripts are missing. "
            "Please run the cell that starts with 'from pathlib import Path' and contains 'embedded_scripts = {...}'."
        )

# Early visibility for credential resolution in this cell.
print(
    "Step1 credential flags:",
    {
        "ascat_username_set": bool((CONFIG.get("ascat_username") or CONFIG.get("username") or "").strip()),
        "ascat_password_set": bool((CONFIG.get("ascat_password") or CONFIG.get("password") or "").strip()),
        "ascat_credentials_file_set": bool((CONFIG.get("ascat_credentials_file") or CONFIG.get("credentials_file") or "").strip()),
    },
)

if CONFIG["build_manifest_in_notebook"]:
    ascat_manifest = MANIFEST_DIR / "ascat_request_manifest_full.csv"
    recon_manifest = MANIFEST_DIR / "recon_request_manifest_full.csv"
    cmd = [
        "python3", str(SCRIPTS_DIR / "build_obs_request_manifest.py"),
        "--noaa-root", str(Path(CONFIG["noaa_root"]).resolve()),
        "--crosswalk-csv", str(Path(CONFIG["crosswalk_csv"]).resolve()),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--ascat-out-csv", str(ascat_manifest),
        "--recon-out-csv", str(recon_manifest),
    ]
    run(cmd)
else:
    ascat_manifest = Path(CONFIG["ascat_manifest_input"]).resolve()
    recon_manifest = Path(CONFIG["recon_manifest_input"]).resolve()

print("ASCAT manifest:", ascat_manifest)
print("Recon manifest:", recon_manifest)

if CONFIG["run_ascat"] and not ascat_manifest.exists():
    raise FileNotFoundError(f"ASCAT manifest not found: {ascat_manifest}")
if CONFIG["run_recon"] and not recon_manifest.exists():
    raise FileNotFoundError(f"Recon manifest not found: {recon_manifest}")


In [ ]:
# Step 2: yearly controlled ASCAT / Recon extraction
if "run_yearly" not in globals():
    raise RuntimeError(
        "run_yearly() is missing. Please run the utility cell that starts with 'def load_csv_rows(path: Path):' first."
    )


result_paths = {}

# ASCAT non-interactive auth inputs
# Accept multiple key names to reduce config mismatch.
ascat_username = (
    CONFIG.get("ascat_username")
    or CONFIG.get("username")
    or CONFIG.get("copernicus_marine_username")
    or os.environ.get("COPERNICUSMARINE_SERVICE_USERNAME")
    or ""
).strip()
ascat_password = (
    CONFIG.get("ascat_password")
    or CONFIG.get("password")
    or CONFIG.get("copernicus_marine_password")
    or os.environ.get("COPERNICUSMARINE_SERVICE_PASSWORD")
    or ""
).strip()
ascat_credentials_file = (CONFIG.get("ascat_credentials_file") or CONFIG.get("credentials_file") or "").strip()
ascat_service = (CONFIG.get("ascat_service") or CONFIG.get("service") or "").strip()

ascat_tmp_dir_cfg = (CONFIG.get("ascat_tmp_dir") or "/tmp/ascat_subset_tmp").strip()


print(
    "chunk_config:",
    {
        "ascat_chunk_rows": int(CONFIG.get("ascat_chunk_rows", 5) or 40),
        "recon_chunk_rows": int(CONFIG.get("recon_chunk_rows", 50) or 120),
        "cleanup_chunk_intermediate": bool(CONFIG.get("cleanup_chunk_intermediate", True)),
        "cleanup_ascat_tmp_dir": bool(CONFIG.get("cleanup_ascat_tmp_dir", True)),
        "ascat_tmp_dir": ascat_tmp_dir_cfg or "/tmp/ascat_subset_tmp",
    },
)

print(
    "ASCAT credential flags:",
    {
        "username_set": bool(ascat_username),
        "password_set": bool(ascat_password),
        "credentials_file_set": bool(ascat_credentials_file),
        "service_set": bool(ascat_service),
    },
)

ascat_extra_args = []
if ascat_username:
    ascat_extra_args.extend(["--username", ascat_username])
if ascat_password:
    ascat_extra_args.extend(["--password", ascat_password])
if ascat_credentials_file:
    ascat_extra_args.extend(["--credentials-file", ascat_credentials_file])
if ascat_service:
    ascat_extra_args.extend(["--service", ascat_service])

if ascat_tmp_dir_cfg:
    ascat_extra_args.extend(["--tmp-dir", ascat_tmp_dir_cfg])

ascat_window_before_min = int(CONFIG.get("ascat_window_before_min", 120) or 120)
ascat_window_after_min = int(CONFIG.get("ascat_window_after_min", 120) or 120)
ascat_outer_radius_km = float(CONFIG.get("ascat_outer_radius_km", 350.0) or 350.0)
ascat_bbox_margin_deg = float(CONFIG.get("ascat_bbox_margin_deg", 0.2) or 0.2)
ascat_extra_args.extend(["--window-before-min", str(ascat_window_before_min)])
ascat_extra_args.extend(["--window-after-min", str(ascat_window_after_min)])
ascat_extra_args.extend(["--outer-radius-km", str(ascat_outer_radius_km)])
ascat_extra_args.extend(["--bbox-margin-deg", str(ascat_bbox_margin_deg)])

if CONFIG["run_ascat"]:
    has_noninteractive_credentials = bool((ascat_username and ascat_password) or ascat_credentials_file)
    if not has_noninteractive_credentials and not CONFIG.get("allow_interactive_credential_prompt", False):
        raise RuntimeError(
            "ASCAT credentials missing. Set CONFIG ascat_username/ascat_password or ascat_credentials_file, "
            "or export COPERNICUSMARINE_SERVICE_USERNAME and COPERNICUSMARINE_SERVICE_PASSWORD. "
            "Otherwise set allow_interactive_credential_prompt=True to allow terminal prompts."
        )
    print("\n========== RUN ASCAT ==========")
    ascat_csv, ascat_summary = run_yearly(
        kind="ascat",
        extractor_script=SCRIPTS_DIR / "extract_ascat_features_remote.py",
        src_manifest=ascat_manifest,
        out_root=ASCAT_DIR,
        year_start=CONFIG["year_start"],
        year_end=CONFIG["year_end"],
        only_with_storm_id=CONFIG["only_with_storm_id"],
        max_rows=CONFIG["ascat_max_rows"],
        max_retries=CONFIG["ascat_max_retries"],
        sleep_sec=CONFIG["ascat_sleep_sec"],
        extra_args=ascat_extra_args,
        chunk_rows=int(CONFIG.get("ascat_chunk_rows", 5) or 40),
        cleanup_chunk_intermediate=bool(CONFIG.get("cleanup_chunk_intermediate", True)),
        cleanup_ascat_tmp_dir=bool(CONFIG.get("cleanup_ascat_tmp_dir", True)),
        ascat_tmp_dir=ascat_tmp_dir_cfg or "/tmp/ascat_subset_tmp",
    )
    result_paths["ascat_full_csv"] = str(ascat_csv)
    result_paths["ascat_full_summary"] = str(ascat_summary)

if CONFIG["run_recon"]:
    print("\n========== RUN RECON ==========")
    recon_csv, recon_summary = run_yearly(
        kind="recon",
        extractor_script=SCRIPTS_DIR / "extract_recon_features_remote.py",
        src_manifest=recon_manifest,
        out_root=RECON_DIR,
        year_start=CONFIG["year_start"],
        year_end=CONFIG["year_end"],
        only_with_storm_id=CONFIG["only_with_storm_id"],
        max_rows=CONFIG["recon_max_rows"],
        max_retries=CONFIG["recon_max_retries"],
        sleep_sec=CONFIG["recon_sleep_sec"],
        extra_args=None,
        chunk_rows=int(CONFIG.get("recon_chunk_rows", 50) or 120),
        cleanup_chunk_intermediate=bool(CONFIG.get("cleanup_chunk_intermediate", True)),
        cleanup_ascat_tmp_dir=False,
        ascat_tmp_dir="/tmp/ascat_subset_tmp",
    )
    result_paths["recon_full_csv"] = str(recon_csv)
    result_paths["recon_full_summary"] = str(recon_summary)

print("\nAll done.")
print(json.dumps(result_paths, ensure_ascii=False, indent=2))


In [ ]:
# Step 3: quick summary print
for key in ["ascat_full_summary", "recon_full_summary"]:
    fp = Path(result_paths.get(key, ""))
    if not fp.exists():
        continue
    obj = json.loads(fp.read_text(encoding="utf-8"))
    print("\n", key, "=>", fp)
    print("rows_written:", obj.get("rows_written"))
    print("available_rows:", obj.get("available_rows"))
    print("missing_rows:", obj.get("missing_rows"))

## 结果位置

Notebook 运行完成后，重点文件在：

- `cds_obs_pipeline_runtime/ascat/ascat_observation_features_full.csv`
- `cds_obs_pipeline_runtime/ascat/ascat_observation_features_full_summary.json`
- `cds_obs_pipeline_runtime/recon/recon_observation_features_full.csv`
- `cds_obs_pipeline_runtime/recon/recon_observation_features_full_summary.json`

你只需要把这 4 个文件下载回本地即可。